# ARC-AGI-2: SFT of the base checkpoint on Lightning AI Studio -> private Kaggle Model

Lightning AI variant of `colab_sft_ag2.ipynb` -- same embedded files, same trainer, same gate.
Fine-tunes `sorokin/qwen3_4b_grids15_sft139` (LoRA, merged at the end) on the 1,000 public training
tasks plus generated synthetic tasks, and uploads the merged bf16 model as a **private** Kaggle Model
that the swap pipeline can mount (`--model-source <owner>/<slug>/Transformers/bf16/<n>`).

**Before running**
1. In the Studio: pick a GPU machine (T4 works, slow, fp16 with an overflow guard; L4 or A100 is better).
2. Kaggle credentials for the account that will own the model: Studio secret `KAGGLE_CREDENTIALS`
   (contents of `credentials.json` or `kaggle.json`), or drop the file in the file browser next to
   this notebook.
3. Edit the config cell if needed, then run all cells. A restart costs nothing: the Studio disk
   persists, and the train cell resumes from `ckpt.pt` on its own.

It never pushes a kernel and never submits. The 120 evaluation tasks are embedded only for the
contamination guard; any task that shares a pair with them is dropped before training.


In [ ]:
# 1. CONFIG -- the only cell to edit
import os, pathlib
OWNER = 'takumuhata'                 # Kaggle account that will own the model
MODEL_SLUG = 'qwen3-4b-grids15-sft-a'             # new private model; reruns add versions
RUN_NAME = "sft_run1"             # output folder name
WORK = pathlib.Path.cwd()         # the Studio filesystem persists across restarts; HOME may not
PRESET = "auto"                   # auto | t4 | l4 | a100
MAX_STEPS = 4000                  # sequences; 8 per update -> 500 updates (xcalibur's whole run was 63)
MAX_HOURS = 10.5                  # stop cleanly and resume on a later run
N_SYNTH = 3000                    # gen_synth_ag2 tasks (0 = none)
SYNTH_SHARE = 0.30                # share of training sequences drawn from them
USE_NABIDNUR_410 = True           # the 410 verified synthetic tasks of Nabidnur/arc-agi-2-grids
NABIDNUR_SHARE = 0.05
FP16_GUARD = "auto"               # T4 only: auto | full | off (see sft_ag2.py)
RUN_SELFTESTS = True              # ~3 min CPU; proves the embedded code runs here
UPLOAD = True                     # False = train and gate only
BASE_SOURCE = "sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
os.environ.setdefault("KAGGLEHUB_CACHE", str(WORK / "cache"))      # keep the 8 GB base on persistent disk
assert 0 <= SYNTH_SHARE + NABIDNUR_SHARE < 1 and MODEL_SLUG and OWNER
print("workspace:", WORK)


In [ ]:
# 2. GPU -> preset
import subprocess, sys

def pick_preset(name, gib, wanted="auto"):
    """GPU name + memory -> sft_ag2 preset. bf16 presets need a GPU with native bf16 (not T4/P100/K80/V100)."""
    n = name.lower()
    no_bf16 = any(t in n for t in ("t4", "p100", "k80", "v100", "p4"))
    auto = "t4" if no_bf16 else ("a100" if gib >= 38 else "l4")
    if wanted == "auto":
        return auto
    if wanted in ("l4", "a100") and no_bf16:
        raise SystemExit(f"preset {wanted} needs bf16; {name} has none -- use t4")
    if wanted == "a100" and gib < 38:
        raise SystemExit(f"preset a100 needs ~40 GB; {name} has {gib:.0f} GB -- use l4")
    return wanted

def sh(cmd, check=True, **kw):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), **kw)
    if check and r.returncode:
        raise SystemExit(f"command failed with exit code {r.returncode}")
    return r

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), "no GPU: Runtime -> Change runtime type -> GPU"
GPU_NAME, GPU_MIB = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
PRESET_USED = pick_preset(GPU_NAME, float(GPU_MIB) / 1024, PRESET)
print(f"GPU {GPU_NAME} ({float(GPU_MIB) / 1024:.0f} GiB) -> preset {PRESET_USED}")


In [ ]:
# 3. dependencies (torch comes with Colab)
sh([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.55,<5", "peft>=0.15", "safetensors", "kaggle>=1.7", "kagglehub"])
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available()


In [ ]:
# 4. Kaggle credentials (never printed)
import json, os, pathlib, stat
kd = pathlib.Path.home() / ".kaggle"
kd.mkdir(exist_ok=True)
raw = os.environ.get("KAGGLE_CREDENTIALS")          # Studio secret (env var)
for p in (WORK / "credentials.json", WORK / "kaggle.json",
          pathlib.Path.home() / "credentials.json", pathlib.Path.home() / "kaggle.json"):
    if not raw and p.exists():
        raw = p.read_text()
if not raw:
    raise SystemExit("no credentials: add Studio secret KAGGLE_CREDENTIALS (env var) "
                     "or drop credentials.json next to this notebook")
cred = json.loads(raw)
name = "kaggle.json" if "key" in cred else "credentials.json"       # legacy API key vs OAuth credentials
(kd / name).write_text(json.dumps(cred))
os.chmod(kd / name, stat.S_IRUSR | stat.S_IWUSR)
who = cred.get("username") or cred.get("user_name") or "?"
print("kaggle credentials written:", name, "| user:", who)
if who not in ("?", OWNER):
    print(f"WARNING: credentials are for {who!r} but OWNER is {OWNER!r}; the upload cell will use OWNER")


In [ ]:
# 5. embedded scripts + public ARC-AGI-2 json -> WORK/repo (repo layout), hashes verified
import base64, gzip, hashlib
FILES = {
    'arc-agi-2/dev/sft_ag2.py': ('7566fae6d314c307de8f0813b18905628ca598de94531490a59b6b7b978dd223',
        'H4sIAAAAAAAC/619e3PbRrLv//oUc5DaKyAhYZKSvAltuuJNnD2p4yQ+sXP3VnFZKIgEJUQkwAVAPVar/ey3f90zg8GDkpKNqxKBwDx7evo9PZ7nfdzvkuI6LZOVWqdZMqz2WZpdqHytqstEvf3r98OJOo/LRC0vk+XVLk+zSvnluhqffKU2VCG+SAI1fKNi9e3PP30Yfv+j2uarZKNWaZEsq7y4C4+OFP3b3VWXeXaiqGoUX0zC3Z0aDqXoa26fKryhV/m+Uq/1865IyqRS1em/Nqf/isejkfo7t4V/w+EqrmJVxeVVOX29vIw3myS7SMrw1zLP3kxfl/lmX6V5pl+oua6AX5vp63W6Sd7Mvy4v4yJZLFQYhihxHW+G6yLfEiiKslo43dHHbXw7LKtkV6ofF+b3Zb4vSvXf/Lus8t0wXldJoQvQ6PfbhB+T2+Vmv0qU/stj4g/r/Ya6rBaHYVQmG2qzrJT995n65sMvU1Wl2Z0q4mxF42VADlSRxAR5THMXV5cDVRVxmr2QcbzYJsVF8oKX8ejob5dxpdJKXexjaqJKklL5cVkmRUWIkGa8+KbnYEqj+xyNZeWySHdVqQhqalfkq/2Sip/fcXH5DZirdV5s4wqQ8ONiGW3yeJUU4f/eJNl35kugaOQqyZY08pW6SatLBgEaukqKLNkcl4oWZzWUAumKBrhKL1LqfDT8aqCy5Abop8ajgdrTuNV4PFA0g7SsaD5qPBmo1/9KtxH9LKp/vVHj0wF3wC+TbIVXZ4S5Q/XjT5+4f+67yq+SLP1nUgzUzWVOaPm3vFi9T64JTQkbh/azKgmBsmpzp1ZFviu5aQzjRT2EfUaTFqgBHgkhyll4q/xin53n+dVUHX+yra2SeDX89OmTOt9fHAeh+kRDMbtCttGx9GEHwCik0lL2JUEovqC1JjSpeNiA13lC/SYEZ0Ir7GkqzCuxCnk9N3lZcgP5drdJsG5D2hp3ipql+RZ3DjQvinQ1IOSNl5jxJr1KmguFkS/zzSamkUrj1CnvTYUdhs7jTPlptttXA0VbnP4GhKNpUUN+PBlRv/FmHzMK8cbG8ux3NGlV5BW/l0UskvUmEVQDFhGI8g3tRPRDNWhOWJQdUFNAwPsAX32GVoQNhm0SYgjRZVxeBgO12+zRodmigvM0MGyfTYVWeUVukvTikrDwC8UbaqXe5z+/HagyvsYiUKH1+CX9WidVkpU5LbqdoZDRPFunAMequZ4KFAmLsUupHYL/Oc13O+D5bfe0rjtaDqWHfxPvIt7yIBH1wksXegHUZbJZMTk9+xPI+ae3H//no7pIr6kXArMsv8DnBVOtV9zCNs7SNcgNgFoAi/LqMlRvFaGt3hJUm8fOLVzklbqhaSYAkADhnPpcb+ILAMc3XORE/S8ApeJVvANhWMc035XFKcJQglR8R5s5y/IqJboSEOf4FrSMfifK3yZxuS80rVDtdbwLpurH+DxdZXvag8VyGF+kw8kQiFuCno4nX2KAjGiDEXGSD7/85f333xxZzBB8u1UoGO8vtrS5GeEYB5kaxttEl4pBALmLyNSPWhwoVB+wfbKLI82hdvvzTboUTHyjLpIsKWIirmpN6HXZ6jEX+rveEDjKnJ6JVk9O1Q9/YSQhENKcCCjY1EwqjnjfJrfUuB4hU/AYi7jLmVijvdPxSJV3GT0SfKkT0Hxhh/bti+W+KNLlfrPfRtejiNAwXRM+8pQ2WJAPzJLLUP2QbHPQCEJPwtOEO3n38dP3P7z99O6jYibaIBF2+eJseYlt4YNoYIizydlLXlTC1aNayqDJTXk1hOfSjCajL4df4oXifa/en75Sn07V6fCcGBnQc5fETAmJ/I/CkyH978/qr38hcvqjAIu2BI2StvvReZJkXAUtqb9++AWbRosbdofzP+LyEa2r4m2uCkW0kEg1jZEK0jYJuVMU5B/lLqEByL8y+cee+BtBecfsSYHDVafycb0bv/zi7Q8f5NdkdPolPxDp4H93SWm4/Xg8HJ/QNPjH2RBr+AIgka//PgU2D7/E/6n5jW6eiZD9dzr6Sn69PO02fzocf2manwxPG62rfxN7RfuTke6ApbBOB1+Ov5o0Oshy+21yNjw5Mx2MQppBeFZ38u/JS5mA6QCcjxaVFgmNoBveA+D7en2uEuADb5Ai/yctDdNVwg/AVHPdeF/ly7gkZsPUYr07mcgC6rUlukp9/ZWW8uMy3qAC4TFWNqVdqJabdLdjpkWYFI5CxdISN7+MQZ7UKgc1ucw3wGdFJCy9FrYVn+fXiXp5djY6Fdqd5dRRvqM9kkJKoQkAs2kj3yYl71G9xRNGR5ZWNMuIIVGdE+UjKZG6HkJUW/HMBiQvrGNiSkT0GA6QVo8gE3NBqWYpW42H2KNE72+oHd6IK83oVwmErEJt4rukOC4Fpf9Vz+pfmKzZwKH6fs3Q49L4kpHkQLpDSlQ6LzQEiBqKBsFQIxnoIhkISZJqtnPZmCI20zb2qSFiThkJTAO12UaXJBgF1BmElH2ll5kW0xAY3sFEoc2eJQDecZ8Dw7vT1Z4aLCuSjbcYLUFnm0PKzQkmskQYiAK6UEfn8fIK7M6ZFPM5km0wwswu1U26Igav50gjKpILIABYMY0ZHPc8YfBfpRBFRKZb5iUkCepHKAWtBrfAjdFvBg7RRm5R7wGRI0gOhsxNYr3QrE+nJLIAFbdMiEPVwBKoFYSktOYpJoVuiD3p5oXR06KnZSoyJKkuRS33O5oeoabRYyC5EOru0SYNlpgFtXgHuvlThq2Y8L4tk5IbrUBqWfZaEjrQlGK1Tm4gnO5LwNwi5UClYUKEE2tA66QZIQ0G8g71hdpHymHchpEyn4OokeU3TNdJFy3AMK4Ai1o5g1pGgtMmicEk0Xd3RqFsHtHFZOeUem8IYrnLCjoyVW/HTLWIri03kMyYBftMFmsJERQFf9+uCPcINauEeKjneUfpdpcXFW3Ii11MspP5DTZrnklNujTPeWmeykvSajf21539ACZ8ZH5k+y0EQ0Kf3dHRf7/7+Z2aURshsCckjSIjUcY3v+PzEn/9KIJwEUVBcETNyjeaNWmEPqlYaCQwzTeFL/SzLFfQS7P8H/FUvTsdTVpFrcDaKnX0y8d3Pw/U248fv//46e2Pnwbqx/cD9e6njwP14e23NGpoddDloOaNz+i/k6P3P/38Nvr09ue/vvv0kUrMvX9EtKd/9QbKu7JP1/Ypt08XBH/7Y7+zj6v8JpMfi6MPP7/7KO3es4zrVafelP6u0mXlr6q7XTLzsM+onpYMZmDepHrPQHOw4BGEg9mnYk8UL14u99sZfd6QmJMMzwLRX7xNt1WgjtMqeDZafXn6/FbBnKndR1oFo+62+l28KR9pdrnbY7RNEJxMnGbPxtzq4UYnptETp1Ea2fQxwP6mVo1pxJpLcof/AJmPHo6Ojj6jbf6H/GMryxGxYaKp58mmjPK1T6oGm0podp73TUurlmLTXtWaBlrk+4tLlnJqAwXsGYYmGzmam4chBZrRN1rl/i4v6u5+ot7e/6B8FiWI9kyYA5Xq2PZ5LFL7ZWKMKEEImsQqXXyOLUVS5mgBE0GS8az4W0pfRvx0cwk1JFWvbYGpNZSla9ge5ulCzWb1vq6/49+v1FKKoTXeSqu/Oq3yONHarwv1XzOQhWY70tYXRCUar8HtrgA4Fjt87omwKs18KkxcxLYfBN32CALzqwUGSP1eLRrfAYFf65nWPRdJtS8y1CUcA1JoNubDvkAsrsguaswAu9TI+aX6ldnsKr1MVmB/12lyA+FUGLhYNaA+bPfa/AEGsF5Dc14R3yd5C4Y1s3pXJPRg5NQx9UjUu0ouiPeCfp8GtEXO83zT+TIJZHnRC1XGZ6dDfzySzzAkEGYIQFieoPYAZJ5iDUd+PSPGE8ZlXBTxnR9DUKl/ngcurlTNFdC145A4wXn4yX6j3sOYxKhs5fsY3ZwaLPLqq5FPNa6CRVjlG0JuH2acxufz5mc9VwZbz2SBGNSVLqVXdU5vgM+YdIoZc+2FXmkjyJilXm8rS8HqNeePMJMD7waaGNBg2U4FKZHEkxX+YlVLUmKIyUMcXoMi+A1FermvoipnnbTYb5y9K/untRwwf8zAnkMxtfp4pOpVcls5Qw4ai2L33+uZmUpzmTRknLkY+meLyYxn8nc+ni5ckP6YZ0TE8X8Dxd0mrSKSiZc+/lcD7lgMFBBNviad/phh6Lw8HpBOeRK8gqB8/DV/1hIbWlf+PkthfGUxUmTGGmDQLWhDJTvaNjHBewahfBkWJJJVKeOD97WnaR8WelcDoSrueiEiTa43eVz5aLOGR3K7THaV+r/xZp+8K4q8aNaHtOtCCCNpQAiW24gnwBAqB8aXMBNgwnQakcitf27yi9muoH0+gLkugro080hg90QjjkoiLCR9Cz+twQ3osvgrXUllAdtU3V8ld1NlsIa08chK68HDK5amY6KyMaR3Ec+1GTbk5u04jllfPWb1uWTCIqUNx8ZXJQPUYrWockQf9aig97BNrmHNCqSfxvwgMJHqQ3qMALZU92yWTldT2sqr5LbVK4FW1FSeDdSAtHqwGJNcm82EycBunJT+576BPdRfv8ZtQ22M82dmn6jc/QN/O48z0vsiKGckclZMZK5BZIwZGhZx6iQQVRFfrqXmBYkMu3JgTd1U/QHYTFSOv2sAzkxPqF+mA5kiOiAlgW2Qgk8OK6SVpWqMey5LJebjV/ADaAQACQmaLQE0pDEXGvx+e1vPR4s2z6WdJaSC1FDesugpr9p0zPwzKlunjSIxLXQryYQwXjFXVM3JGVaUsgbsrAi4XHbHs7KOAp9AmMu0r4Ul0G/LB3sECrMH4tUK4AueNyVZ3ZCWU5t6BPTzRVDzQbsPg2ajAvxO3TK1tEyPqR4skQt/7RlUugcD0D+CB950PnED3hcgpJt4x3vGcdfAMKdtGQbJwXGJaJQ5fIq2ufn05eLBUFVmTrqAzNgOseZBalKPk0hTmaiPd6SMb9/dppXvZYlYfImB0v6diKIvdEL3siOpB6JLVW8hdAySzoCaVyyrksBLrwyRwhuPaYKniL2xQ81SMT1u2mp67OiCBI3kbraJt+ermMQaJhNAmE16HpaX8divDAsOgvAyuV2lFwmLJPMp8Vh/LBLq2UjEU7QYqBcv1GQUBAuZCBNmJhNTjR8YentSFSM46ISM9MEM+xk127U0WpCqMRWk4CEAJXgomhrzF6pH763PSX/yagri3Zf7LUtY10FN5rghS+SoBW2KsvahF9rQo32Zjp+kZs0uxXdwRTgp98CMbKAX/Kj3q+az2/gqibbpreGyuoDUrBnlx5j0LTbvIYJATPya4lrfx/HXN8eqJHGdbb3EWuc+aa17UtnS60QqDjT2K82UymCBbQTyJMzso8zK+EdIM9DiDZNWkvli4YTylq2aVJr2wDW1DapE2456YJMcm5lTEYOI6pCQWdbCI3ZIg1wH87GgSLlDQwyOhYE4UUf/1pJslLpFoZugA35LaWl3lNyJbBlnNwoytfdjGtQStyhywB3hVYEMBM5MbAgg1q0zCrR165J6iylS4w2s+lAFk+FXhsg3yrPieUukZ3Rgbh0ypNfemP7YY3ueqDdEV7gxGiIWgFZrCxPwWCMvm01lApjdlcyZ9Dju8p/pzgfYSGYL7Jzq+TAGAaS3nQkLzfIxzaFMOtDqPHXxAkK9j64Hany4x0Wzk/ISvVxJy6NQIFNeDkTVRkWN0aivK1d5pWfH3+wy4D3pFqNHCXtuNhTs4PBCG+8rgTMmql6mvI8E5J7VUtk4xKoqVINRLfscGmrDdkGzfOOOi5VSao9Uffr0AiNvfHMVUyoHbbOhPMKnKYRFxOrkFoQj8Q1ZqaB9lMTDmpojmMEti/I1yfk2IcmKeASx1ZQlOF/qoQ2SzmMlVu2VOJIQ0VM71xPHCwSyke83K4IryFCSZJYKkD4suruEFYVafojovT+ve9P8SO9os6GZNVhwR/W+JRbWgDFNrQngPbUiOhMUcunbbwk1urMMeqt/JRYIhjf6oRbR8J52LL0MGHrz4XhBJKxu5Tou0phoHkRBjHWOJud9xhLeJ0ANIjSLpkY70MKkaexgfVMgMMRKCzwDbWOzdgM2TIjwaARZ3mI9JqSWdaFlc3P3/6Pq+lH/hlsyUqCF830Kx6Ydo0h+bIvQfdN2++OMqRK9xptE3AVw6Pr8MlqlhUae63wZnw8Ir2bdKBj+BhNEXcmQGv7UVQ060yf5xFZ+mLbjrLYpB3NoKnMDw+1MTNedsfz87n9/+f7nd98GWr9babnL55GEFyQ1VrSSmr3RcjKHQ5NQnbZQ9cywdWHYPtMHMyG0qHUk8K1GQWKb41HNWDtj+/Hd395//+O76PtvPwa/AxaMZat0vUb8TifCw4nW40GRoFPckdxIA37wmuvB4/Xc+DyPWNEpz3R8+vyRNSL8NPqTFjc+Zfy0ZhOuUuPGQC3XFwM2sUZrWgCSOJeXGh4yLTdmTzuy3u6r/Ae0AGN7vC/jzfsfuMYKqHAvjpaptBWeMzWDQ0h8Gua9+/pk0nx9MnmY07jmHjtDPE14xGk26+0+xGAjIhhMzogEOzPkhiPxq6yqoG4rlAC0cF8m0TImDKHG2QhkFkiDZdrQ/VkE1BFncRFvwYoIU1s2rLAggpESG4rYYePD+KJNINSB0yKAvEvWlQHu+7yIv+FxDRShRoRvsmq2jgFE87OvA1/rBvxixlAsvAWsYEVMmvzuMm6/hDpKvHlGEsJAnadxOSOBI0u8QUc9f/IfoR8GRSPZb5Jy5ronByyyROLd+ubtLx/fvo/e/+A5rE0bVhz4wjm7ihpQ7u/3MxNLV9qoiORaAiJiCYZ5we5oUNOOuaS5VD0mk5BduzP9ICjq213M0LReOW8xba5TaEJpokY8V5Rk8TkJPw3ORdJdXFWFWUlPykQcKxrpUfIgSZ9vjlO6OlyeKGk/3HTcULI9T1awusNF3xioYqsCYhJdWU4Q8o/he+KX5GCNo+8+jF9GP7z9fwRtDhwioVWIl47LiSRsw0DIJVdLkpVYq9oaTNq2MUnjpbCVLGQdz/dCL4CAxGYOad4Tb1uJSIcYgslWdxVmWfgDN/KeRJWgYciHIsFj0NYPSCUydswv4kgkM24t+pa6WQQxXadLOHO3Owg1xUWa0X48qyXe7yRWSVuVpLZxiNrwJqJ/OpAF47+Jq+VlX1iTqNOkhN+LWYeEjBURYB3EJJbgQjsmDESmBNf4thkJtTMNLlAuL8tIooSoMNxrwqbf1sFRYNc3iFOC9CSRz1rlVO9TDlkhgKerBGJ5O5ZqYDytpO5XKTsEOQoKQDgyUYTsCCB0LTUISc8TLKp9uhxTNHsUnbQLMCZF8DLPrxiniDYaJ7BUCWCAlOMcWGKGtZ86uxJvUd2PCHmwJTnYu7VtwS/hUhODm4NwrLz51Z6WecDWwyAQURhKVNtYy5iZlpFER0H4Yac2XjGtArnh7ez32GRhxq9Iu6mIAfqBIW4IhaH/A6eD3ipSzkuztSe7CbIyfKpmNLJy/rUZuFS47rYGSLOPnHeQ/jVQTknjUCJoWp0qHVh8dYzuem3qWfICGq1UkB/BaSUM8zr8L+JVMgvY4kdMQqSeJR4do/0BQiE4LYuxDUU3D1kMAalpSEHdZWkMfPu8QevNMlMQNrR0XtqwbACYaaF54QhCsF77xrLquPMkXl8mlDMnaU+f18KocuyrEdrUnZEdnX4AWPowhgmJ0V8GtvE+UulMHAF4TTntEsNhMDbHcknA3ObXLut14OK7IJMSnxmyEW+KJF7dadLDx0i0sLGN74g2XKRVuiU05AD0pbBNN+rzlbjwHLomlkeJEHR0pHlqcPy6id/YHY0dpHeev+UQtXrTsYGP9jSkCRk9byovwnCjcpmTWD0QYVfvz2v1uqaahg03OVyTXdD/Nd3CoMXINjID0vPjt2y6cTmJ0NYWx5CHB8syaZiQjQ0aXLXWv+aMP++zJnsrYWq8GmhXpgmhZXzTUbR23eKCEHzLXlNhL7X92ImfFX756RkxtANlo3qvBq2AWg6n5QhnfV4BEgv7kIgI3NiwX+nsm7zECDgw1s9yQSKh7wrRr6UOl8UZFMuQqJwcfNB2cQmHVefJXZ6xNdxGNbNWJ/Hd+e7Ossfv1KwmX+t9xseKzAmXZ7FOyxAxY/+2aexqM6rboNc2c2uYEJtyD1BMQa3bo34yfNtmnr0doWEahM9jdT0xtx3ec+uwepJQIj7uSYJ6c4Jmx23tTtNSit1rfeOoyVDYqOQScvmsiT990L41mud2tp2q70I9ILwwrKbKfQ29QJzwbHAMoeFZdwVDUd655YPWdIX3Ya0PTDhfHSAuj084X4VOpfaE8yLVXMvM/KghXumX/ufxQH3++dVNqzOHeZkzEL4QD9FC5TnEMwmOrDqtZn2jdm3YNCT/83kHY+IFhnCfTZX7KdME/OrG2LMeGkzHWVAzwY7wgWK/V/ToTqOJvMLwJd7nd6tNsL57oLCIH9YE1hOpR5P80DN+2KyOmaNGWbaCyVdzqKvpIiD1EwNyRZAm/jWCDGtxXA3VlatusTr5THWLCs08oIjnBAu5zoYllBKJrGdC7LANPum3Fr/fp//+/qN7YoD0MH7FVgPoW3KO0J50MOcLLf3F/sxXTOy8fL32Oh7Eew/fiVvyZ207Yz5J77ChHzrtwIblNAQfYYe5jtrM9XC/3Fyr45H5bTl8JuPYFdzbb9F+zfou2ZJoetWL05kuvYIwEWnLIisQxdxIGgtIHdyz8TVhbEAa8zOQcV7BuEP1avFG++yKNClbob5X/b4FKdqIv+0F9dUjoF6G+90KYp6tNaPiDmRnjqND4hFq04nR3++vHhAfRmhYqnua1LFUPV7MrxbT8OTiASdqqBK0eVaOOWWB5u5UOQzVoCsweU2hn/b7Tecckn+fPRhhxHOiH38vEthwpsbC9Ic2ndOUr9xaVxzHwnE4vDJv1EmLFXWcPi4sazOHHBVzZHY2LNZH6RxawKYWIRPDYQtknX9e63iRpglsaoStpICnoQ7vs6YdB7JAWwTKtCAER+AQ3uwe4L22XmtDKKjdubOveuDbxbT6/JQIyecJJNj7UXjm6A7TcLQmZOMDdPfo5dhu1WPBRE45sNtrquo1SDtVsG6L5+uDMuhbK8ZqQXNOlQgs4gKQL5s8uzB1Z+5WvOtUps6eW/k3iRxui9YbYuSQejZ6CVL21oqlXwy8NKnZbRDKt0YgT1eED5cFwEgSf0ECvy915kS4p8TCjbw9UHd4NZ7SdNMLogEwJa+S2xnOQBhjJoKisCRmPTiYqOmvP7gwXWNDTvVAJ9n3MDp62uZgHXra1Y5Yq5ZUdMixTEXn1YJ0VoReHvIfN33I/TGUvVGKiOP4wtjFfpMNo9FMVvMQ1yoRNNe30oErGaJW3KAKtv/z6TrpW8PGsPGrqbqubX3L3d63MS/Xj7ld2FvZ9JMYTR15DiJJ/2CmCy+LuN5yQlN+6HUw6qQRGqsdh5tWS3gwXCgiXhTtM3gwNSjquvwA5aXpb7T+GXO4L7klJCh9PSQHaeQcYVhsSadP7Henl5Dn6DgX7bQg+kVlUqTxJv0ns4tZ7egDYNd8SKIMgYBUwzegOeTIavhmdH6MLzqZMUxKjNrz3HBslcXSOeSIYzV+vSTrBnM1hWA6IgGbagb6qHal1mGSrUpsR5+kfSeNByQxUhkz/rurvEDqrOGy9mTVnNIh0xCOHfCae0kDHiaICboeNMdcI08zjqjjyucXdXEL4z/0kBuvfb3RcNimrSuY46fAQ3FJrS9MbIQ+TDmPQzm6v6hx5EpUKBNOQkAtWJXi//PpPq9f645ZnGTz2uGAF7gn+RCXW0s6dyJM4lDIhQxqW+nYfpu8yG+H73NiD96PpLgTNEL9KWBaYX/WFkAtUINhDDSGOuc4Ynat2oMcA7SgTxIwnPHCxCMPusca9G5Nb9mLoKNWTZvoFIh91BdzRatxLfynwWJMANtAh3wiUI7qDRAY6kSCSheLFgcSgQndqHsEzA25F5KH/vRgjljUIWi8he8lHPGhEazdGiK1ZJBqjsa9xVNxqnp0es6tpeBmDEi5rRrC2vgDHmWlIfnpe8v9KvZqax5+wqQXX5PWLh5t6YiPyspEiMsBn5yADtFMIfRpMQBNkTq8pwEgto7WDn+CRtyHE7+i0VVHr8Rhi72AG7hiGbMy9uPtHgvh6IkGWDix3zrnxr3Zwg+0vNT/w6ua9ZqeOL57F8Juv9EsdicHI/A5mA6okobvvfw1YUH5rrIg51QZIQ7M/82XmnzIlwG54fgNsfRFq2QZ3yF+w0pVBEfRJUvtV0NGMLjzIkkh8+KFrIeQl4WJ5eJjj7o4/IXU5AlSP7kNamJcEvVYtYa6KSJ+vd8kRfieTZTvf/Zre96uGhjL5X7KqssYcp+/RwAygnExAkTniibhj5FZCvxgmZfi4til9B4V94PWqCQg1R2mGScnNTEDJVwM61wnfq8N0GqgxI5ATYiXgEKxT5aliCi/csjO8qrNbXVZDwEhzCBlJBzgeo5TcZcp8rFcj0B52PyCCF27NVnLGphsGc5CPm3+GdSKXRzq1A7gzS0haHnlyj81xjFBh5diS8x1Q/oLizS8lQ2ykVybbe60nbQRLlzNPcIpj88ng/OBDtE7i3TyxcVCXULvXvO9E3V1IEaz1p1vYnj8YaOD7iBReXBu6Awz5Svko6szVyEhF3ce49S3JFp0aG5+kxmufUgebjqJjfSMucg+9RbG4NucBTVNzFjCiCD2RP41aJV5rW3vzn7hFRGBPuIRoY8cYUZ1Md5yvQX5S6NoLwpyUfoCioJnbUXAI0qZ5+uRV4fvSm4V1JUARv7tBXyc73lGSiPMoKK0UZd7RJ7pWNS4gYYJ46CJrbujzPBbX2jUo7apzYSUxxWDkWg8/f/hxb2D0IaIJxvsv+3OFTOQhwgnr1qh7wN4eYXbaZVU+LsWBhccv905/kGbxBrUgu6idC3gXLLf/K3b0kSH5lrHp466OjAjTNcGcGDsj5n2BKqt9Hs2lol46/VoGp6uAdPaq2j3vLuzKpEvrohbeCH9qC1vQtWgvvn3ZmtOD2jKyI9CG2vK287ZSPggG2mqt1rjY6+VT7bTVO83zzrM6Zl31FRvP+wpUslhQAfpnIo845DMqUsvD/RlyOfUFYT0hpwaZtLB8Gl7NxDvqNz1IZZRJLtNTEJfBTwhniELUY0GnEloZ85AACWQcyfE/wAvnZkE3w5YMgBj0me1fYfoNM7xI+rU4atib+e99toFw7TvDEF3Xwnsn9xaDj6WJUuZzzfdHLXCMHoiWVoMAKESnSQdzwGG0xPEaT7442xcFqjZSyLp2/qJ6n/N2ud5jBl59DgpdnzFJByO+oguH9DsRnf9RidIg5I97Q4ZmGxhUVwxcsz4BFA3kwnTGodsTztJzYZvnDhUm4SsypuuEcfHYfweQUtdY+KPdX7DwDrjo1csznL/kHT7/Ms94f04LdAep0R93pumHnRKyF5/RGpyZ0r+Ue/peG7tOYNdHjt8Sl3ekPA8HG4KcPY0g6JZWUfFwIk41QybvRoxiVsOVJoR77VQHvIf3iaqpY+ECJZh534TuCCbJiquNrVak2bQFHWaW41lVHr5p0ZX7CfqHd0+44fIp/3ZHMUF+8ecWBnYscoQuRIl3B/fI6uzjcNRZxMfipfkpvsQpC27YGpPWhCdrI6S+YhlYACBnXtY3JKjcXmmvUtEhbsAMBCS/Rn8bmpm+Cn1EfSsUwznRiQRzN1l6mzqpiwmFN3fIPTq3mnpIVD32S7cJnHmA53mQ+fjdBGw0KF6dgvtSN9hc2pInNCcIOVRDIUbwiY/DSfUiKT2dHbCeb7PVjFmcxAVXTDY4lgt3wKFzdE1VESNom+zhnbVwiFXdHJzCRGvizhrraksqQJNMlt3skTR3CKfq5OXo8Pj7TJumYbbJ1vaQpCaSOLfbEUNS4TyuQWC58xKtr9JBdIIKXLxJreJTR0cIgp/zEM736+IEx6zkdwZMTPBYzfD/fFD8KoHW1z9VHJBE1EtEkmXScTaZF5scY/akc3boJWbwgnEzNJSZFItbLki54MIatfjP0JWv6pNQrBSYstwNgabDQvmgvKCtamuL8ha6noNJFLWC3pseCI4mgzcfKwsLqFVtpM1GtM1shcST2IxWJiTttjxG3lsydlGxmZRsbuCjqgMBGnZzKg+UTUdwGrFdRHy+0VySKL8ikXSbn+1OPdb5PaehkQY1vG1BesbwiwbdG96tUAmNLc0iGV/4SGCs3q0Hc/gVyQKnFFn7GveI3g71m/TLSmj1wwJQtDXKH5oBmINl5Cdqgh0A+7b65jXvSD6QOXzK+DHVU97psQ2Kcv4gsE4X3vz+831w0Ldbx881rA31yb6jfB5IWYKdnSs9tudb5CRcBkuj36ENiFd7O7C2G7wf7jAsmo2DhxzsqR6YOHn3lSbH7eXDqES4Ec4rdNbCkumC71q5BixmrVRpLmN67E8E5kzC8E0Ti+Fpm9/++nnj+9AsFYcrazKy3Tn5N3YxVmywdl6on1e7chqgK8dPwJYT09LDe4aEL5nLkv56X+mij0LuY5R8XAdRcwZeoZpRh8RHfgoHQmODlFJyRZq14eDxeTROLI5+2xh7AoxTAomJ234trjY4wT6B11GF0EioyjW33xPX+LiHfxc2xl7PjLxGnDkEWyuIuQeLq5pG9HxyzyFH0wHRGhHY2ATY8+QYfXwiM1NLtQSx6NwnjRT9XQ0GuFXSa3URolHG2PZwDTGhKRujqM86tak6MHWNH+sQcL5cKKKVvJwpZov907I9l+LFAfOuV4mm50UU77OQV5z9ECH6/NVDVscYLOZ+fmVTv18eJjaz3m4gE7A3D97MwnDZg5DAxynDw7jycnp4VrExocsXx5CCgtFK4keHgNt8kcaO7NtWRn8cfwSF7nbkEEnycVwCJmcSoeHWjQQ92A5kQCe0aC5RMnZpnOdelBn1Fo4+1Q+yISsZ/Yx7JREfTNJvn8geaC+68nJIfgI0tms6c0B6+hbY85HCHBj3PJd42QtxDw6dM5FLDdo8KimnM6fA3HFVI2DoibE1JyS4fSgDevLK0kPO3NiEp+YH7ffvyleNiZhCsqA7TE6e/p2XzYufDp/jCrpTMxPEDMoy3owpd/IYtlgbPHuD4tssTmimQVGuEdLAkJWzVAtz/Pemgu2JkOJNBa/vL3Sxya/IBhzwJKT+oL1OqRqVhLJr+BSqzNuHcw0gSoTkw2Bf7RTTnRDB7RaIrE3TgOSHiQq038mvNKX6YqEMvmNrOXIWkOdr1IY9/ktso5n+22kS2qj40ReIiwmw1JygHQ5Oz2ULgGFr5K7iBO76cITiTyUxE3URH38fjYeTU6RV3IVMRCjdDX78PbbgUrysn6DrPAHujt3y0mMfJUm0U1erNxuHP9tG64+gS7oCajrhMGtngiAk3wzM2jWBc5Gq7TtxyId8qEuauxJ997f96PxCKocct17uEwMz9AgbOZuvCAweq8JVG/w4wQ/3DQqU1x0pt9xEnG8OdMncGqxXtyVG/p473F0Cgma9qYzzp+PsUFbwd8HjIHQQS8PKxKLhz6VAMpbM3WO0Qe6I8C1Tui3deWV93TDkc5h0mzf5GOWve234tOS7Q7Rfv0xa7wX3fA9/mTKsZfZVv7MpnLng63NTO7IJCFJ9ghbZa8i6+isuUpY279no3G9Su5X2+Lfs8nJ37PTs/5i0sjLJ9v4s4MLR70poTmRq5P+vSejs1wNiExzNh8en31G2rn/minJFz+bqTlh54nc1kBoeCZ3NvwZfxZuO0gfuOEIQNS0yZA2OhkSAs/Y/cRJ2sVsG58joF3iKn2bVx7KkWSxQEN6fdwkkc5tQZATphIMUacqRhFk6eT+xbyIhFuQBPR9XjpZuMm7/mSetJEJv5L813N/Pqc9THAZkfiAUHN6nCzwPOcnehjjI8KzqSxB70QXHEupsfl0Kr8JsPRk0vMlkgHRpv1t5V47dXOvEWAgNHQTi9UlZH20WtWX+NZNeUstcc6pZxaXJGe1swKZ55AV101JjiYbKMdJHxPkL3yjTkZPhGr2hcG7ednHo2g0GjXal0GThiv9H0qPjmnKkjI2muS09FbO452blIwECklS1jsAW5HPqvT7L+B2FFmiynO+NQduOpZrOee/zqTrToK7xWAeS/Q+5zMOMhv3Cp7GAE94Tm4S6yN9Rp8g9GvPNTJyMYzEueo01q0w2W5M9q/gJ09coOfZWLb8P2q2eQ+sbZWjS39Ti3Wy48ND3cpI51cmRErnN8UM3OxuiJR0rkTYzT0+3QIfIIQwImrNr5I3xnz+7SmqbCxmNReZnppCYuXJaDGfno4WzhkUzSfDTwmYH4mx35obSGnocakcwzzfu9aKAlxx5EfDE+oGFOJT/cURvfG/hhWc2RGOUVnyklwjf/hvmPVnyIOFZZOMt6u0vNinWn8x94nWUTOsRc74uIgsGCLIbag6L+1Ds/TcwwCJoIw5lPheg5aEo3u9nlNlL6LoX+6gvrji2atqsOFg4xZb6tYfahRIrucY9sJiwmLw1KE9sR3OF/X8l7Qzyl/7Fn9ppbLOJ7NbbCu1NKjTRfM2RNu1xNiSGsHB8wOro+uXnfqc8n6m1p7cxXm//PVhel/++uC55tqyvt6kjtpvBuzP0dBCzkCghnbKd9hnjRfmDHhV6LPh7Q/UBYe1ZHe+8xHmZCcVD3XV6UO7B4hUaJeA4UUMi/pACyQDlPs//ETl6oZI7Dgf94YRQWLTDOFsPAkGTxdpDy8eYzzUPM9uDCGx28aXrTYaiQ6KfSYZodjKKI4Brdv15D2Iiwu5bURs1Zql460/tzZqOSZD2CFW6YGkC67N0HLphmtnlijg52/Npm3ZG09GHjfYsC565q1rJfRIfh25YavtQNs6n0JPnIQ55kLz5Uij607shQPFw4f83KToujHGdQlf/zyaOtmnXci73uxZo6tHqbO9LZA2EaxM23Q1hK84DMNmHLRB0mePSZAQ+M9ngbVHZCG3RZvAXxZd67GSKPzYxAbyQseXcww7Xy7SMzHcJl9fsXVbmXke/R5Y135w7FXeFW3Cmo9dase3OI3ntdtn0d6c27nrPRSo0DvjbWXAvByZ1x2Hr8RsDNS2p922XxWnr5uvtQmYJGCEiMF55oIu5ZsccE+LX8xO5apL9477oKfPlttVd9n20dppNvyxC3ca9nxlH4Qdj1st2tRhAAcO6EkVITydsedXlkxrzRrexFvjTeRrMK75gCGWSbpqBOGv5RSdaw7pGmAOW06mfZumKwqbOaxhainOvSBESjCfOU6/4NwtS2/q7CaEOzdjw1r5DGZPC91DlR489wfqmDH21mrD/WYUIr23zOBmbH4wq9ps/JuxPZhQp2AyFsFa6LgZ1w2TRpAJ6hjZn0uMmkGypHDF1PyImgcE6E/Q2ZumJTMYSAYsKVzVx8DdFLNB3Z2uGgzM03x60mm/NRINLzaSauueTqXkyRgfK1DP/zOxJBf7HbKCfdFIRU8g3GfuZ3pZE7XJQaI2gRDpMv8zJ6LwZnIYf3TtesM+Ey2KSYdXYAUaEBvL2k2wdg1UALrrW2X1Cur7h/ebynvk3rDDU5fm9JWf1hE9m4xagZXdU0J6IAcOBuFQZYKF8Dq3k9WNQNVLeumD123Rk4NAhZ+4GME+KoJnMqz2kqkVt3DaUZ8cXPgTzF67efX0N8XsJBmedpfsxOVzPWytXaCPPz1SxmVWE2dudQpfvpHZ/+bDL/bO8WCKa+A21eWdaMjIFlLWKT+0AYl53Ca9SpwbpN19cXoQPKcAj0ioM32NqsP8ud7po9z/dO46LBeyxfX51DqcvlusJ2HKoFOsr7cDEoTtph3u1frsLio67OuiV+44fUTwsK13xIFO+0gLa0o2JYphr6AB0wp1MRmo/mqD/lpt9IoFfW4432GZLPNM3w9uPcMlI9S0dsZiuyGJRzXoxMWbrAlyTxrfDNLMx37Q/cEeMOv+kDytVZ+QRK8dTHQSHXCOCeHqVMYh4odp+GVePUW1bwxv0gnXxuF2swvtZc6WS1En/nPL2iyMn6uT5Cw4lGHjcPf6WulndN4u6XZNJP6xru2K+DctM8tBsBHOkY4IPXd274nZHL63XeU52flWbBVtHX23J9qFTxw8z+dq8m2X50tHzlnCD708mbSTRGhSBl9H5727Ow46v5n8EX1tHJpngHCXOmS4ZWLsJOqShNpuZLDoeR3W084yhbs3rDjPCdmcVK/CjNw0bTBlvlbs/drVeid8oi1DlKSFc083HhriwOaM42YOWac6ya/syOsvZpT1N7FQMfkRA5XjCuTuWGJZPnOZTiZ9y3QyObRMG17XeSvREACBnNg9q2U9TxtJ0FzDc8N9t5tC192mJGDgcFuOO7OZjFifXhVHKW1d4TbxrQ9eckt8484couBkAHh7p/uhhb2zTlDMkAYcOCzFvnseqPP1+rftCD0nB+GoiYMIh/CooFc0OOoxxjSB1FwC3YukqjJP40Xf2gaWUdr77+QEG7KGkGBaPg80OojqP4IN2jgIHA4aC3q21KRPxGiCwzT8HHgQPhFCIMNXG6n0a4M+DqXfKN4+jPgMenToiCA29AvH3Codmk9ajc2+pC/M5QOnuKtNZDhzApWpCWshRTIkQrRBPkK7UA0XUgQX/cB4k6yYUBPAs4Oi8FlXFGZdYdLQFbbcwNmjMvHZY8KuJYIo9qgwe9aRVs86XTX2AGp0ZFdDLtoF24YuJ4/my4MwetlVo58Bs5uzJ/Trs9+gX9+8fKKxl79DWX/5tLJ+Jsr6y5ayfhaYo+KJxdLHlHYHMxlX24K6DjPl3EfHX/PVgsdKLv28zG9od9EslI1G5eS5EJxWAyZfnCudjyOm/0xe8UWt7T1W31rK0t9d1id902s2620ccY0tAOIUu8vEKwbFft29ziCrhTVIn13XwGdqfDvWMU4SuirRMoeCbTiAgVP9c/r3UsePIq8oMgJf0Mpx8qBzXCKQXTiWATus8KYA5lvvX+nfswcvSnG4ARP+0+h05ak/IYFzn/N1Ps/o41ccI4SjS9ZxOp/7Gb09CerPD7/j5qfef2733Ms4eHQQZ+4gFs8bheOShQfQ+7vrVeUrUiVNFtxZa4+xYnpP8Hr4ehROzrrkr3VPd4l7EzkOpFEZCUKouo4eaV3tzcXrW+sd6aSYwJM64Xxkk6ZDVd+x2+P3cK+tF8X9YAa0liCGXsyVzs6l6fEkMBexukXG7jW0E57FiXO+k/b0xE2vZi8F1tMJem0FVEnYNtsa/nwG9st3zTKVMiXGugROSusCtVqSLuUOnraqJWLqWa1rcUKHHo2LNnG9zBkstwhRkGgqN0EoNdlJEAoVjqUJFzbsRHQA9+Bo+gjQKHH7Ft8yjVRVf1xHo17XN88ocDgzEoPKu9etAb1wxC2+MU8IqLlzd8cDgBbDQKrTlmEJOn2PwvFX1INt54UiSLN0dQIFuPm6QT4JrXBI4d8nJ1+a27HvEhwsI15VwHFqsuQJpeyomxjOfHomYZa/Fy3Oug4Hi9t99ILPyFpk74j2HXN2szGQJqI4X3ldKnTSbPkJg7a5nSQ+z68TrPh/ZsGub2HuNVwbw5sJ8r8pcpKDncB+AkORrOUohL6XPLszF5DQR1C4xBVyV/1hUqsnzWVUpjcMpid8+y8f3rmB2/feiEO/61DyL/88eTgQyd3DdPoce09Fd/eihJNwszGbAyvdhvV/uNS9l5K2l53zmeJ4vI4dh2NWJEl1LwJlFJGwAxd+FBEIv/nwCw4Y0Jo3XNNTDjYbwjq7ieWcvQSeScBm88B86yxs/yHYiTkE23v6lb7yQZ0VolmkoWYPqNeyM7dabWoU+uMr4whfpUQLY86/0nZva9ESEfBun977/Oe3Q7kR094ppGXrV66P8AvXQ4jU72k1RPqZKl3GGznfNPzuk3hXbvLiqnxVu1umrWk+6m3xNYBPn4Dw6SEQ45x+1pxk52QWZrBGHojG0ShmTPo0V339EgNOZ7XZ4i4ztMJfqlxN/jRodtVSU9qQ4ntuiV9GEYJxo4h9OFG0xUQinXk4PhAXFdRJEw3iO/kR7TEKR+8okUqxMiYUcw2XzkWg0zPxYWl+grHZzbhoqkfRKl9GkQka5vCX4Oj/AwsgkRKLogAA'),
    'arc-agi-2/dev/check_sft_data.py': ('1032c05f9fbf48c7ede726e8e685bf9871a977ecd74910c7bd57f1d7c138d316',
        'H4sIAAAAAAAC/7U7/XPbtpK/669A2XkXMpVZyUneJWrsN27jvHZe4/TF7s3N0BoWEiGJEUWyICVbUfS/3+4CJAGRcpzenSexKQDcXewX9gNyHOffa57E5ZbNeSnYLJPs4p+/nJyy67c3LOIlH+HYipd9Ns3Skq/ilJdxljI+53FalKxcCDY8HTCx4QkrebEs+iyKi1LGkzUu9Hs9Bj/5tlxk6TM2XYjpMixmZYjA/XzLXhfZWk7FOQvqJ9/3xyw4ObmTcSlOxP00WUeC6b/+xyJLaXrF709kdlewq/EXcJycFCKZlaIoe70LptCwuGBZKlg2o5eJ9NHr6YIniUjnoiA858EIqEpoJ3pkDIsvPvzE8ANz8/UkiaeslMCNOJ2zQpQewcPpZPR6FieK4iTw55/G56zzh3ZxF5cL9sdsnSRhKe7LP5h7xSdxlK4lg70MT1+OYGe8RFxpMZVxXhYK1Vf9gIANbET3H+x7eALm/MESkFzB3GKbglxL2Nh0LWU8XSfrldfr/YSMLQCnUokaptgIuTXoQtaKez4tky27I5JBSXKZResp6Y56vRSSiVUM+FDrcMlcxlHB4rK9K9I90DfmcjkNk4xHQvr/vhPp2wqU12frQqBIASkp5d/ZgsvoZJpFImJlthQpqCZPIzZDnC+Hr0577ECpYZq5cZqvQd2zdQl/PZbzWOKGptkq5xJAEefUlmkum1lGsFagSJ/aG3EjscrovYJdXL1hyHb9keACJABZq5zH1jnQTghegl0tRCTRzpDVyMUCN9TGwtMt0JuAnrNcyNW6JJp89h70Hdg1XaCqEvErLpcFgb9bZIkgsn12s6jNjVQC9t9GQpJFK+PzUzQyKWYkASQX1Yqh8TPLGcBbBV8J0DdQcAkKAk8Jl3PQhAKEJfpgP3+uRQrWiUZYLkBiijk5LtFzPcdxevEqz2TJ5p/ivHpe8GKRxJPqI1pd9ZwV1ZMU1VOxrQfLbS6K3kxmK2RbIkhNQRPV7E/ZOgUNA78mZnydlFE8BT+iJ9P1CvbOC5bmvd7Plx8u2Rlg83NeLvwolils160+80mBf90wRLcQhp7X+/HiuusNBOT1fvvw/o0x+TGLU/dwJQIA3XeK9QRsCWURAq9mOUrZgXFQYwdM9+b9vy6vwl/eXAO83dOnO5CJG3kjFpHxRQzEBTo1F+5w4O3htdvUGbHhAJ5AphKfh/DMiwKkydMSB077tk44rz/HqxCmZfn5HBc87+sxkUZq5MW+2yl9y4bPgLCcR32WomkhPegENd3EVSl8tEHgnCud17c1stvP55/VR8CDH5DizzWpn2/Tz7cR8ODm9w9fhuTaL3u3qes//YdnwgeuAoBrr/fu4r/DXy+vACA5k14P9IOBgoLDcZF4b0QMAm29gU/s5Fx5IYZOrjJ2thQyFcmTwnRWm2zKJ2swjK3PPvBYuTU06pIMVySF8NEG6NDLCvJVQAVIK1AnIcp0hTys2OfP4jSCw1SahOFPPGMrnzbveuybMwQ3ssQqET/7L/Br4lLKTLozB04htF2QEuAtYvARlfM2KGfgHHYAbT9iO8QZIOQa1TgYnQ7G38i90xxhAMzneQ5MdmttDVb+XGbr3B1442YlgILtrnxcqkZhGzgIGwC/cbjH/9MtIJrvmEW8FOVapghJ60B9vrmNCvzWdfy5BcgTvK6OSPAkkWtQjOpANI5U9ZJXyx127NTHgJCFw9KsRJGDW/NXgCsRhiRLubXFWjk+AwKaYZr9CRHf2+eDYb0azgEB+H+hF4h/NiQDX2BTNAYhkWf1rwFdIq7AX4EDnAr3Yl1mN2gL8Schz7LJR/C4Xq255I3IBfYZukBbXSsHKO4xVnEt5xihj2oChHzreB4d+ZHJHVzf3gNBgRBDgH6CJUWaHjwTGpAV36zQw1QCa8It6z1eQajpafUoMwrwXDrZ+my2KtV0rSvBQQzSVxExOhA7kKrDTkNR3ITrkAKjFnyhjmkb7QGcIBz8Dfw21NXcCsz68D8kJG6wc4gocOOxOlbUh2xPMoOxDPlLexr3GRIRxkVY4z67keuaAxBIFUIx4cBRNvuA3Xbz4Qdl0AUqQ0yxJsqWJ7ABDOOAG+jDIbKovbgKLOvN4+7QgegDgXwjUKlIqY3LUSo1s461ndzfprvJ3jjWaPsQHEzozELQ5EgR2AMuyMH5mnY2TQSEnnXwA47AxCohMru9TWHzDV6FyqnpJb+nsP+NnaJY4bxwJZLiqjPcOr+9IMZ140p4Luwg9HALAiIaISEZ1OC8B3dBrIwySwSiJQBNporvz8xTKgTGZdG25p2BDLOUM5r1ixxSVBeDEs90BYiU3gZAuP8d/pKekgiFNABjT+IY4hpFZOJKPy6ieB7jmWet9b5w9tF+G6lJcFsQNOFhwQge7dCgUemdPtSCIE5Ld7pQOKcLQjq2CRhbFhi4BCCIwaD0E5w9w7GCEDdBG/gr3DqtASs59ca1ocUyxLgYLYnsqLG1n2EYNY3Tqj5A23AZo7k8mHC4sJ0khigFzQMyPOVg4ZWOhKOyuAlmOWcMvRx95kDwZAAjae7zgkvJt0ggkG4MILG1npTNZk8NKXGISidDAOTygX+DQP0bD1WjpCgJh3GwkQjCWjawnh9InKMdE1kyK18NXIS/VGSpgQkN2FoiEj4RCeUuGF3v6zDMRDqnQw3hH6Ck80f8WWnJ3IdMKBftzB6BbBDI3Jd8IxLX88sMkzO3A+ABUE2iX4hS5zDuRqmMnvE8G+FSbClOzqULYOw54C7JE8zgigookpa/ptE2KVr2sMRUbZ2r4WaHLi7xdeDsef5C3IM1CdxZFVRDXk1qLAq3qdDQqdlvkmXzFAXzOqMM0MeD282ADcabSLMVOGAWpWKHEz6PwyaPDw8KQk7FqSJrI6gp+Qr4dnWpBq8i+t2+NoBlX9nAFIKUUqwKU+o4n5MLpQgsTp2xLQeAFjSuIA/0UQ5uBZ71UT72MF5bWkDhYMgPzgTAAJKB1Y/FUGTBcgwuzADfCpgxMwlVEOwWuZj22Yrfh+gQDyIjCAq2sUjgFHGxUhHGkS4QQE7G7yiiCDMZ0ku191nC+d7Hk0YqVUQMIBtZxsh41xk1pyguZWdnEFpj+cZp9jhFWRIQSuHJOEBpO6Ac1T58zzCzTgVSa5AShYacmNaCSigpnNaoDQendaMYB6LBFE9zk1x1ys7P6pG2vSrhWMOqBAPn9lHV6VLBtg98vDpVQZhGQJFlt5cj2io/d5S+tjsVCYqbUoLsayHXKm2DJc1EQ9UaWR93ClelWlQNNlQry4G1WMfyUQdw21L5w6jAKoHr+PNPjqdUAVfUL1INIctdXA7ORUKYhYWo2ajFeFCZJAZXbfF+1s3zr9GVI/pihGeI1scCYH7smMIKbJyuRVsRTfsoXITUFmMJGTtYdCkhqJsLiBG1W4BoN+2QOupUXWEntZLoq53mYzeRFK2DzSscDQSP6ukaMw10v36YfrfVplSuzM6L+qoI1vWWTsub4BRFLx6FRR2bM+f3q98uPlxfXvz46+WI7cTe6TCSQjxgHI90CfKIS7BhffdIYBWzwWtgoOW1/EQ3EovLh+Zp7lIF/dfbApzp5T3kCDNnnS7T7C6tyjRkyYBphwfJN3LPsGioG0ioD6r14zlV9LKEZLg4DMCbSNMMgP0i/iRaUTCNmieng1V0B3fO0aVMlHNwdFGdJiYQkHE9rkrsjiaH2mP6mBUbHURZ522fbYScZIVO2hXBYgPkwq94Rr9VDkQBICExQzTPrDEcFBdSzOl4FJKq62fdToKkOcvCJIMM/owNenXfh9ozAu3cKMC72GnrriWjJBj4BQgW6q4kU50llsHOEp6rdsYUBSMKioQL5LpSDJIkYNN1f9dDPVP/AWm1O1iKFY5iLeyM1lQxleSnD8Q3hlNM0QSauhtR4fMocgGilfYqe9Hht22eDWctWEaNwKDbAxV51jZvY0V1+s2cHRBR1UH3B16u5cEx/zLKG20yO/1htiQNOKyNeajhLTeo3d8l/YH4a/RAy7Mqa4KfGwyG3XjfclDiXscBli07UppGZ1ts/ipWf4ndRlkvimczLNJSQfJoP1X7wUY+aUnbK9Yr13Q1yqWw74jO2GMnbAgfTLdjrciqFS/bxb4GF1pRtQXE28zUdg3sIorOmW6eNIRa4RYiVdCNoFpZa6Dc6VOaDk6GmFhYUmhRaPN9QY2muiwCC1vpLZVmxKYtL9MdBSCiMdmn2ASLunCDjmPnKFt3RkxZu4OmDp8gDqsyC1Www2fs3q1Tslo+SfCl2ophRsk2XMUFNW71rL6TYRLooGsLsRMVqmY3VmpTCCQ07xGNSX9YEbJbApmZhCF341W55kZFxjRqvlXnGHsbuRYOgCPnrD8iTtJsnDAU3X5XkRuuRBRzbDvOIOArUV3VCCpDoZIifFLHzcAfdALJX70wIeQCxADeKRGu8vGvXnwREunMcWpSrZfkIJQrruDsKx2AyGVJ7Q/jpIMjjUJq88CrjjkApU9co2WT4YkI54SLxqsZ6mPhAE9Y4OzQaIhJlPTMuU0BJIUlpo+uZnVNdZfulRfZNSq4V8fND8xQRLarqd//UF35qPRQTapBmD0MHWcOqpzALhaHQEkb+15fw2A7SzEhgNLNLgjxJYtklhcgIB5hLR/vS9Cdgllcep2b0jDNGwIjpoTFdiiMJ5aCPRmP/MFsD2f0qxf2PAzoya4Nte8hHGAx9UbDadHraDdWXRqpq0vkkkbMAR8La+rew265Z7sNMGi3Yd+TSjxlw8GAgP/NczqtVWtKZaieFTuYtjw6OFFB3ZwfLNwn57sn/SdqYOPtbXxUeyRl73Ar4xp/MHoxtj1sI7qf3l/dXLz75eri5vLNSOmjCUurpVt46nqILlU3zXPjvg1EZsAlcb83laSdvzRiMC/+gGfGQNZNs+pWDKi4wBTYREFT6xTbgDJTFe7vpZjp6yLwqCrgoLUmDdSLL6j8a3jAI1SpOUhs4A2rFQB81uF7dZOtyqerTq5Y5XirRCUXBbYxq7sIVgcLWzO36WB4m54+a7pJ5oq6a3ObvjL6XBSH3dZkB3i9Ay+F4CUR+KefTvvsGTy96NP1j+GpGn2FQ02IDK6DquFdhLGhgRNLppOp4z3QyNZ7BJBeR+/GSOMcxEphqXrF7NO0kukD8QBLenbWGwS4Z8wKcMdjfAjgD7hlmILdv1BD9AQPyCOYVSzQxQQr1PVMuR2WAZDzCjW1WmDIn2Ju4jqH8nne4jEhA6dy76h0xJciT7AHbzQDq1s+eKYcrlKagkvoyfEeEoZB+CMEsuIJXRWIzDiXQPwV2XxbN7CmovGv1vU+jPgOk2+KdoNgSKo70LKCx1MSprGCxmB6qETZa8eS1NxReiLkyoL/n2QWrypLAVhgHaAcf++zl+MD2VfgmiaU1YDySNALUgUDN3W/sPnVMYnkBBxLKvgwGatF3VhpSYMawT7zqlcbKvT4Q5CIyd9B1POU/hqsHDScHgIrqUF7SPOkz7iaeAR3JhrEF36+NfuYq3VRsolQbcyeWeI4LGR0VfP/H7pH/7uG0RLbqVUIsPCCgfI16qjUZ4N/I/Cw4HL7JsbedSa3LpWMjVDg2/o2Ah69IAS8wDlfg+1Gqpry5vLde90xbs3dXF7f0JyRtCEoTIlUCXDEjIskyuzQUzb3ScjQQCv2OEpVvoNXwI6e43TTFMH6+XQRLAfjutBYbV+lpXw5PEaCrdZGAtxUIj1vXHda+w9euG528RDYpinRwLV2a2wNaT89RrsBtt4+NVMCfKyo92++mnoDMLY6BgjQhHOM2vLj4W1VvJBVakWtl5HeR+tV7u4c0o/BYACQ6BHW455hZEg5MoiuGTrVQ6cQvZPJlB9h8s4xQlxJuoAFTpVojzCixmVi01Q1qdjTvKN9i+wOZNHD7UyikCE2TTiyJ/8lVebXZ7J34HTUhf2RuhffZ3cyo28LwHHHwQ77bM7lhM+bEtTHpIuXyugT6gjZfaC6geR+TJApR5pBPn2nwq1FUIAMqpYJpB/zLMPWidHhGFE8sPcwhrDu3zwGHN32K7oAdgYYDB/ZM8f7i+hQJC1kOsoKXFIm00M0vYav/xrFsZ8OJE1//a/uq9imuK3KBZCh1DiOuMnhoVsln9pBgGExqnGBFoM69CiLocIW2cjzygDMWhbNDFUrMSsr86DPKn3stjkKKwIlT8uYNOKj9NZtjOEh7Qpzz8i2qkQKS7w6J2gyPwwejRvZcYTV/MaOIQGERPAE+5qwFD7MF0eLsn2rmOB0h6VYK7cuYH0hwyQWTjWxB6cwfcXEwqnShuqbJtSYimcsDPEbBGFIXekwXIE6haFuTHOMJrYFhGzzTTAcjXtGSZw3PgWXCAzmwzDKpmFYX6TgoPoEt/nqldHybnLYNiR9WatuQ59RFZMH3I8h9b53neaLX46nr8NhC9AYxsRH977qJp+6UGOBsb5epmFpUPZUFzwsslEqeF8VnO/t9jr3Kn7dq1vvupt/cuKoK3NurIqC2LoNYiywj6tbytYm+22CPJ1JHglYqwKLUbsALRJSFa2ozCI23p6+mhOn0+oLSFQUxK8xKStUsDTKvmqT4OU27AnWaSaygYpPyI5Rh1Op+oxGy6tndF7+46xyCmbChxgDfHPcBLXHXIXqKoZUvsJKKe5O17GpTFV9bbCunFYqChox6ohJmg1TUI7XjzHEwPuRkUjLs86yq/VlKXeH5OzrvqPHUHqlSNGudwCxqgtqR3Sbfri8/v3Xm5FT1/vZ+39RFxc4pFq4b99/eHdxg9fhf/z18t01Ozmpbt1W37Ui/zNL+HwOfkD3H8yDzfnsYNNf02Y1VqsiG5BXa1jPNkqTGODA/wDIAllMODoAAA=='),
    'arc-agi-2/dev/check_swap_model.py': ('1e6331bb368e94de3811a63e4964d0973f4aafc151d012006aae4e35a33314cc',
        'H4sIAAAAAAAC/5VaW3faSBJ+51f0ah4sZbAC2PEkJM6eOHFmfSaTZLLO5szxcHQa1AIFkIhamHgZ5rfvV9W6tADjWR5s0Zfqun5dVcJxnCstpBhN1Gi6SOMkx5c3nz58PL56L6I0E/lEiYXKooXMRxMxVVmiZv8Urz9+Fmkyu2uLJBV5mo0mfqsl8Fnc5ZM0OTH0Ar2Si2CehmrmL+7Ei5FMwjiUuQrCOHspbl5kKlKZSkZmYPAgheNjrWZRrnTear0S3Z6YiHen309FtkzADjjnleBZ5iJMlQZ3uYji3EgRL9QsTpTQozTDXEeAHTFcZolu0fxKqensTnxbprlsC50aDngruFMjiHknhgpKUdgVz8I4GeNIoxJfXE9UqzpiIrPweARmNLQzVYmIQy1cmY0Cnc5uVUbCvPr0OvjPh9evLvqgP45zcHT8DApVKyLR6nbaYqlVJrrdtpBaxzqXsE631xYLGYruSVu8+DOeByoJ/3wpuk88FgeC5Vk8ArFQ4XxiMU9JvDmLVGpoFecTlkznKWT8bQUWb9ORHC5nElLGWrz/cC1G6Rxmj4czJdQtCRHRnhbEmMQ5FLKEJubkF0rD/K9JXZDy7aurd+Ic+mfts6ba4surT+8xOEvT6bYOvT7szlqK/wtxBfPVfVLrMDSzGmct8Mjz6vsCDGAKisX2UZpE8Vjwh+UINIhhGLrWWAN6z0WD78eCVmhxq5le5YkgtlLxeAIN0kfLSOU4OwWZiZIhyPWh/0zOMZzhgGWSt0WY3y1UW6j5UIWs8yxdEVtzNSen4c8w6p7xmUKOZZxoCrTeqfj5Ai5s7HF9fS3c7Lz35Ey8Sz+9EnEygl8x0cBo4PFsHhAXXstxnFY8X6RZLsazdFg+f9VpUj6nunyCSyxHefXtTrdany5/+3z16fINbLJ+9GiNFW7owRE55qHVRGQyGSu32/E2beGQIzp9dkWn8kUagDc6tRvSyJNN6/3ll3dX7y+Dqzf/JvrOH8tOtyNpEj7t/JHw00bYnx/E8C5XxzP42UxcfLwUGpE002U0IADE5x9B5VXr9dufg18ufyfKNw57c0Dad4gzy8CaBiZxGKqEfYG+JkuozwzN5B1MWQ7KHDbO4zRh7WK4ZfPGS6bqLriVs6UqlwgHYKmyuQpjQrTyjNr76BtDYxCWDM7l92CR6piPqrxFO4PWm+vfP14GF79fXxqVvT07hZaeYs/bkx6eTumpe4Yn0vhF/XhVLbyqFl7Vs09J13j4XD5cfPjwjh43rVYrVBEcX4aB5eWB8XIXgT/hyBTGOdOFSngQNLKh45FFon6lpwRcGzfzl8lCjqau8+I3iBz5dID71PNuOoNqdaZgooS91Z+lUKdbrEs8r+CrCMJAL+dzQBJ5p7keKPZ0G+4S2FEXUMQBPMmLNHiBo63huO/TBCs6vDOKZzwFKQEcLoWNT3/cVPskmP8V158LrHIe+ZZCHA888X5ERkSRwYRq0Wk8AR60Rc5xc49CI8+Hb8616/Ub3gVQpd3i/Fw4QQBYkbghZRA4zWX0AcrlcbJUjQnSfLcxQgxpYiW/cfREwvUGu7QS8QiqaAwb1Yofz0XSGC9VfQN6xpUHAxxaDvtjlbv1FNTtiR+3SBR2IdLikbCcfWfzqbdPOz7gRZMfuo6Nh77xEsfbla/2CrBaK6L0wsID1w4bEwExg3vzswcHMJpw+pW3OaWwGKs9z6kPwbjthw7Li0EjeBlsBhzSqPZmMLfjftVd6FOAOEYhpIg0rxar78Bg7S4syQuRjMvTX57Jv+KIKtBcE8aFRzM7dFUjYXDzr2wLA6kOhY9nBniVGagDgV293CIBqaVJsPBmYHHFu32tcsgvlzM2Nvkx8JbMjW9x6Aw82yi8pS3uZYjtUOIEJ2kuZZZtusTPSfDieNyRxy+Fm07Bkss3Sxs3stZyrLyBRzcorUqXOV0lxi0gCWUpcj4MpSi36HGfVvlyAeWFrluPe155eyXpN9kXlz+ddFu1aiED8QqKleGJ0cqgRv+UbIHpWmVgwnUoi3J2nEHMY9y+ybhwCjXT1r6hJObXeV+4RuscW/Dn2DM2wxOZrbz7SzyqWCk2iH+ci3hTQ/uM9JM3SFg3/L1UAGhxDfmYBX/NOLUFjZw6UQ7jCNmYiLJ0LqycuUqYhctr+2KcgqEyFUT6ssYRG8ezD6WgSWb3n+uggCmTDMMBagdKGDu7dEjBlGTvpZdOaynIXuCGMIVV4m2KFBYeQQ6xZr/A/SRnMzvPJeE5oW6eXdMRL8XZ6Z7DKb3m460kfoIbusECQi6LgcLHx5zymsKlUJ/JgvUEldRUjDMowKLkds88Cji+g3Ygy8SeY1JwG7JG0XgXe0YAH5JpC8gwzN5cAxfWEIHD0WEdejA0DDELz8oMDXfVC9yfTw45plVQrEsqR/XgEVRbFZtccVQ1Hwg3LVkxkccqWKVZaGeB3v2G3beeVIOMS+HuMlUBzCepsG3ckFxV5BlVHGKYUvpWVbaE4DC6s3N3VomvnoMIiraxKalpNbKTBVU+nLZxNNBoFGc6P86lnmJCIzOUlLzDAVNzxlBqLoyb2gBeN0XO9rpMw9uwZ9vZvGbKQOhBODgFDpb6ngIFs/qLwUPWQFVOWNaZMgRa6zfbOQmdsZtzUP1RnHxrTmjjAYfQchsn+WT3oerFctQtDpo+ShT5aA6gGguKotigqTZwWtW5wq26Me16lDCU1tsguhtNu7hXnNWHz1MhNZKzslCvD4QrOEgMabUxZal6r4CWFXS3nfY37ksKsNVNkbIN7oEE4Hkjfzfp+l5UGFPyA4ImWRuIx6L36NFJp7UP1wvGoJ/VzZFJC49oR1c96/u9aCMuyvwZpqU1fOzRYMPnu5RWrsfDvt/F0p/jC9JGGOtp0TvQe6PQVlbkrKeb/vrWOvLCaThZUdewPEWyOiidzo6RAlgpdLHWSmEHhCikYwLcvfOIin0w6h1CTxuNzJ0E5dRUSUOcpQ+XedHG8S3ABUf3YK7lodRaCUpjFrn7AHVGb8eiPyABlBoBRhf5aJLiliYnPb2wA+Ovp2whouqJhZJTIFmWLhNKCvyTY/z5iReEy4x6PQSw3Hy8Nn0c67BxJsMYAWG1V7HjuVhkaYhSOcY2XLhafLm6/teHz9fNZYYotYhi7hEBgLX6Jp52n/UaWVUh/EvR7R2yw7pYWLsgfIbG+iyBRZ2loLR4P/uYRtZSdlYtDi2LAOxtzp4dSlkOMIbsY0zX6UQmu3ZCGlN3aItkhq4qTuUOXGpHuJvgSNQ89H3/1wuT2NINCeGOjNZPzV1GfeWiWylFtITYGGneYahsmiHniWOUldyh2RwSmxS4oE4kNwbddRW+RyWto4G38Z43xaNLl/sx9BjESXA6jPPzt5KQnyJ2JJFNHZI+wsb8pFeCFVCaz4cyTSIIQQut0wmkl0STDzx0c+8Db6zaqeazGsBNw9oO2X80IvieC6fS4FYbtoHM/faG7/DKW9ZZc7Z/QElc5DoLlRkvyOM5vTSQ1D4a3om/tm6ABmHC5e+FrtIpdEJmlsmdKRq5yWPubHK6opAMyOERb40imKpWjBVVbvnao2weFa1cgPuClGnuT1o4l1NFTQSGynPTCwB+Fk/cBNHFF/axc+OpFoIj2yIiITI6N6zNd0uJDbWJY+QIMbMfN9vE9VJ/uaC8wt3u/B7uIi9kyA3kk7KljOksN03l0z1t5po3zhfD5XyBE023ANe0U7RrnC9Im9+RpqucCqOm9IYQt9TebjQw+uJmgMHd1HNfbwZDK8e6XEfcvbUSOzDwbaWSk90MD+c49OLl5G2avZZLLWfvfqWWSKNr3Re9J2edvb3rvjg5azaqGykjdHS21YEGL0OOf8KmmuXSXJQVQCPrvZod3aeRRiq+rQ5N+bDxOiZtNOPv6+FBHcQwiUtKMAuNqH7HX9F8r8MfLMLfQS3BJMzKqoF41S5l4GunFLrs1pkWYF9Qf457rGkUAb7ZEiDaoVzN5FOcxOlJmT1tPB8YAobcWrK6Kb6jEcN6o4VMetltmjPO+KsMx7hF89xqnVPpDsk86qbS/1bdjC/j3r9WBATA2jflG0qXD8nttmC0XbHn1MNXkQXpDBsNuCb8mesxdU9Ni40Lr8YShLCi90xT0xQB0N3MDa7dYi+pkAlQA+OWwJhS6YGhWjsIv3/cwx9POJ54+PNDQaRuWPTFfKmRnsh41pSQF5bgSO+9uF+zjjfOHkyDk/G7rw7DTQ1fT3/qbQyorrfi7Un32cnZZismC8x9yPV5b+H+mwNmKETYZwjuDxtj0I2z01Bz2CQsZmAZyKt3WOLcs7bOuePxPrNh+G8ZzRju6UWfrWS6T8Xb0jGwqGk2EC0V3sTG084z6PvvqhjqpQ0DBikMnDbh5IDSmYG/o3I7NX5Y3bYwD+k7Qs62R98Y3g5jDFWX+9vG9I5cvPRATDOXjUTV4nNflFN+w/lZTTGPVbiPcxrfZp3GSkvv65ChVMmW6pClDIWHRNpH+/8WbB7r+T7BaHxbMBr7u35KNtmGAcvdjJxEsPFKFRdBfrePHZ6wncDK73juwClmvjxmgXIXhX+Zi0Kjdr8HBFFUgIJ+vgPIyB3ohxjw89D8AMX6rcRjk4qRQPwTI2OkRnZuZOATCC2eg8PkmMuW8qcaK5kl/EoI1goCemMYBOUbVRR21dvUopmu77SPIufWEy+EVTTTsPoe524QhOkoCKrGU7n+pjtgqvXvkKy3tHWOvkuwU1YE22heE27Xp/QGu4y+FL26QV6/hyvfRZXuWrNjrBU5N2te0z/Vm4FYz8u+SWHM8qdeH37hNiKCxDQR6ec/UoRZujgGZVSH9s95ql89mfZurK0GQUG+ltwm2/Va/wNJbvJ5cSYAAA=='),
    'arc-agi-2/dev/gen_synth_ag2.py': ('821044d0e93f63ae4138295a1c6cecbdbf680579f36388bdb5db2032b7cbb7c1',
        'H4sIAAAAAAAC/61ce5PbxpH/fz/FHF05ATZJkdyV5FChq5xYiZ1TbJeliq+Kx6JBYkjCCwIMAO6S2fA++/WvewYYAOTu6pwtWwQwr55+d8+j0+n8mKVLHe6zIFZf//SnXl4cY62KIL/N1X1UbFSgbpP0PlG7LF1nwbarVmmmPvz5o0pXqtho9fVfvuuN1CLItdqmoY77V1eK/nbHYpMm12qtk3l+TIrNPFiP+ruj6vUSdT0YDOgh3ReKy/q/5mkSq2mvl2sdqsEMj9vooPJgqyeD/ptRVx+KLFgW9DK67sZBttYZPQ9ezXi0i3/UT6h3xWakBv3rV9yvPizjfah71F+UzJ4C9k5n0erogvlUi1zHq0LnxdXVu2C5UVl6r6JcpYlgVUUJow0zU/km2GkV5Or7YBGFyT57GWTLXrCOeqPeOovCXP3CnesiWr78BV3lXZWnapEWm6vlRi9v5/mqmIdBEWBsj+Eb/2EVxforXwVJqFBsIPMIFVRR1StlOghVVKh9stwEyVqHb20bFYQEAA1+9aUKo40OwSIHtUzjdJ/1djrb7ougiNJEBfv1VifmJZX5reIjgwqYVZDRXIs0I9ruk56prsMxo/KhA7zMo7DTVZ1VsI3iI54Mv3XGajpNd121C+gt7/f7s66Sfzu3cZDnqMukRM2HKNnti64izqLfE9cTBumAJBeqnK5+MaP9AlIxedJ4j+mM8XSnvZL7ubWvJhPTAYuDJi45EoRRprxM95gyNNeFw0CrLN0yMg0e/vrhh+9VEBNb+H31YxZtoyK60zmhl/iCK4d5PA/imEmXpcXLVRztXhZU8+UyS3cv97t8GcT6JYF1FxXHl0TPeK6TZZzmOnx5RfPyaXS1pze10USAIlV/+/q/3olsd1WSFvjE06MJEG25gMT3L/sgI8ov9lFc0HypEnG5zpi8INnn1FYmuw2O9H+xhJbgD0YnDEcD7nIvLCHaZJ+EOiN+ED55melVrJfmUZgqStZMLK/J2tT1fBPkG/+t6KSGEBMeidMSiJgFoAsNs9sv4mipuA71bSeIGTgU2+7zQgnzc1sInmWPXErDaLUi2ANCfJ5zK0Ig0UaJkAImgmLY718PpPslOFNBgxE/6YR4ZCn8sEyTvMj2S+EtDCdVCe7A6Ie8/jkiIMC7PKygx3AeSVWg7nJhSX7zuyz0VU8k6Ud1SwpQRehJxVScWxUkQxSboFD3pINIvu8TIv+3pLC2e6JpRXgt/d6jKn3Vhx1RDtxFOhCgETLzkjk+bjTZAoj86LXKCe8xSUSaLAEFi3dEbO7t4j2NfZ+SwdE7cP0uzSNgJSd5oC7EojAj7FJiQhaMnCWDkRCoOEp0AJqBvIRdYrRVkCmxDarUm9Qw26VZIFMgHAQxdN6RyEKoIDXWY2R9/PhR0exiEPh/h7p385aHAZ5YOLZRntP495CkrQ7yPYSYcLEgXXGkOZAY0Fgeq3WdZaQUCMvCZUsaOCJGhqR/ZFEgdskJNdtdrGUy1Ok+iaNbTeSiTgsNy8E0EsShSQoLK1AkZCX76p2QgaDepkSEQOVb8GS+iVaFsOEHku/on8LWK2LaQmX7xEBNqA12jBOZTrCItWgFxuBYjSBDvRuWJJFgY8be3ygCAgRn4GmerpUR0A3/kTlPCJ3kIBQRyMfSu8p0vqkZjbfqQLosWuyzF1Rnk5IHAkDBla+veZD9DgjE+OqVhYfQ+dd9SBxAROWOMc2I5Qs95yWlRZOBXXUcstcBijKpRRR4CGv2tkESrYiruygRSRndCJPsgkTH6gu1Tqmf79+/V+TvkEX18vtgNyeIF2l629+GBNgPELxIlBG34iG2KVQ84SWwbArmJEHwQIcsIppkpYz46j7NaFZW+Sbrt9QBcQqbVKPomN9Gr7n3UrjuyRQwJcmC0wxJ+2xFEe8C5pOVvgc/7XOM6ihFiB25JJkWq8AKS3BKPhSJCyGV6ELivIqKgkQe2LNW6sMP7//+7qcPJNikxgtr/Kwv5POkBco4JeEgYF6ga3SHMoJSMwuWhpJwS2z9QlgyYBUcLfdxQb4YIU33rzqdzhWJEOGIXRr7nOb2ieTyyj4n+y0xJ/FTsiMN9+6nd4oMeE6Whby6MMoSQqNn34NFjl9vPoePNJ/7/hV1JWXEXjorvEFXoRPfdl83WBhnmYdKfUbA/yMYq3c3g5GtahCGOt80alz98esP5wCTod79/ev38799999U4aEDupM3w66x6hjnmD+Qe6w6wl38Pnh1Mr7wZ4a41ka+GRm7DxXORoT1odiFq6urz4iAv/VPpTvjOeRXoV6Jok8gnd7aF/+P6HgDA5GQMiO+wcx64g84tQE0GfjeP3WWqqWO41z1vlJTjx+77JRmPvmE5BORC5qRp9EXW2n50+mK7AWUKBw3DDu5gWpAMwY1Jd+WHBJN3L4kQSCnC2wOK5zoaL1ZAK5iT5qbWoneSMVtEDFFp0GWQQzRhiKnvg9ORc1vu+pnot66z36/NIaCnBBX9jEzQoqUdcnFT2Ofq0BfTdRUIhUAfQRAGfwV71uDQ1t0qIp+dopYTa94sCm55YcZoUitzSO5soNxK4oizJDK2etawZIQXATL264hwcR20iVS4AEkmNZDMmfQifqYNXq835CISadnYKBmywM14/L+Lt15frsSIOkHOygoz5MmfrsasJNQYcIoQkXS40Ou3FV467lvPC6VO29U7vvjs9EmoXag/kBUPKo/qG/Zpsr7gd5/5ncoMcaEwDDjj+vyjWiwHF+MZGsNzyCxVpdRZdEhjRroIIYqK+SkknQoYuQjuim8pW/qZ7rYZwmqkzKA8M6Js27n4q1n3pqYNKJ/KBbVmcGM8OqapG13NLSC4IFT7EiuAuiSb3qcxMF2EQaEAApRY514y+lgRkX4wZNfMn4EaojAz332r0its3/K3eYNYQDToRY3GTdRUPIkTYK9YxYEpWNys3hKF3FwzDF3Us4SvljtcxEDIY0UHlht73cdzJI4bUAz7ISkHvDBvsd6BSUO89ID73Uy0jj2y9A/TctBHW1gZkkqhJT+PUI9b80M5UCFP644UeD7kJ/p93BVF0RmW3Cx0UOE/4qdD873oeXgSzrExTBDUiu9DEsb4aSIU1LYpb04g2FW5YYxmOJtK2PklPmLmR0wD9swcyFNG3APLkC0WKQH+AbxM2FaPgLTkcoPkI/pDshGq13JtNCmO+C68fnKhXgbJd6ReH8bHPArag0fD+bjQT4yJS7MiB4RTj02nzPGq2GMaNhvWU1etElD9FGvwSqiMiRNk7W0bCb62ajK8uMXZz6CnXo83cbHL+zHtqZ1ufUS1ZF1mbNjlRcloqx2q1H3qrRLVAIKcK2WsnN0Xc8oO/rPeRmS8qNfF55v+gAD+uebPrNhuvJY8kktGcYmhWkhTvdJMV8ErKtddWA6o4arPXEx6MLAuJPwfdOGrULZp6P9OQ/yuABQ1L0A+bnquMa2rKMCVsAXEL7K4PwC9MuDoCfSjWP0g2eo1vJl3EUquXzuPSoC2whRO4YLDpGd1kp8M6TeYirzwZIoBuydOOuIuTA19qGlfYVfq4//g/iqq6jzVYnJTRCvMBxp3uXGjHdGyExfD50ihfFYT8ffqpcv1QgJ0EVaFOmWv0LypGA8qyzJGvMe/1w2sPaEv0MUfzZNTlMGY0bA/fDjB9iqKxMywB1HEkF89SxFc8PEBPztmBiRPmIit7BYQEStRoAa+Lqx+GPkbQzupOwOzoCMQJFMku/SXDu9oAv5zk5DxzAh1bD8SB9NIrQ2eIiWpgAgEg1CO1AtYVprtWSQ3WLTdonBhVEwtjx1LWbiuD42m/0xuABlnv01RTD44aHLPmaHWR1d8gNm6HhaPE3n3Q5IjogUwYZ0brXeNWfhMCD0HjOgOB5iU9HcPIKbrFnD5/LFjGYsBIrMY9dwiAk+LfWibYtw0dazA7tKFH2578Yh4rR2m8nKIsNqMh6ECP3g13ZvkdYgaFtxClZ8hsuqSgbKvtgpSjgtI5q8extAU1BKQhE1amVE9UwYAmWo6Mk3304H6b6gmJ8TImq1Yefem9IHVynNzogVld+1arOCmtmxdkF4hl/oKwBDLIRUdUCIQCJf5xNm/tssTebI6tSITO1Q4BlF5/eDvDjutAfDQUxHI56ufvzhw0fSKx6rEKsouq68O4LtX4huSB9RDaxrIWssSXObQzZLNFd//vpv373/7l2lxGyyZPqsoR1N0m0qiUr6u0bgS9nt1n2KugB3jbRaMXUksCZ33UrOzLqVk9qZinA1hagmNV0jEXVRqPG37bhMEU1LlrZM260zYld4pUb9GUgqhqyxQIYlkSqz89FZSxN62SUFyTyOFUWisaQZzcqY6Qlp6iAxixvotG/TKGsxykEuXo+s14RguAl7Ktb5YK6IypXryv1odEAGb4q68PrB+5/zy3AMx6zRrTHG639TdowzZPmSMCLZsblEOmTN4nRy01WbaDK8MW5YPvkei4Uyiw25DTQJhOtZH/lWImXuxSmacN7CxPIXCo3rGuc2OMvZ1zZPOWcrMJrokqy/3KTRUoujaR35IfQ5HINJY6BRV73C+JnexcFST/4cUCe+SzlJdXmYhG+8S86OU2nWpwHCdGtKfQo5s/4+iZDD9gb94SuyI/1XprfpdsZNDHiYkAPStp/vt14jmbF2/FhycwnTGxLj9F6gZLx/KXinn4RRzsHUPI9CPRn+G9H/OC4McbK+s87ewr9UZSyHZeDCiZIq5AIYCaibnCOsS7TXvt+I3OZVN6NBI2bLCOLsDBIsrri/7qOlzVgv26ivJmqD4A35SeqdXu/x+sz0pMkqNMYkvw19Uu/nKTPo8iA0XEUcByriMorgqNJRIlcEndJXV5kSSQ6OEWFKL7N+kFCE8kywKTw1vVJXpheTNsmnkfqd+n0rLBam5eBWsHYtz4yy6/a4Jla2wEsszV8M0BIwGxayf4azvpioYe37ItPBbU2qAJOpzZwFHivzN6kImhuAnhegN8TVb85QyHx/htg0uLatmepMvp42agyFV86xyVDYhIrqKa2mYmF1jvlWm1Yqc/i1a9CwCMZRl/ol3f0iS5i/SAveh3JP4byiPnl1W5LISY98gLsoiEtjSIhP2cp5TV/FmaeBsaX0kEl21Nu13+yy6cU0PY+a61L5GH7d1tpxfTcTt67pI0yPBoUL64YhZ13A2yRd3mIKWGPB9hReSKQhAuw1yGusCptivVL1n+RvW/tiQmwYmEH/ui71hsckdNpy5Lf2m8hcO7gC2GWAdBnr5xGNxhLLtVpa0RGmQD6o0a6M487i2xlx1MY98N7Q6TIsC28DgZPHETh47TtCZcyyKzrDwUXL/AhOnXCwWbWKtyqn6bqr3vgltgYtbLnR4TMZ9HlIupwFdDOBXGvatj429YbU3czm+1p8yDQop2aY8uK+x89YKLZBhn1oZjUVaz5Bxnso9IFiCnK7ecvU4ldtwvdnrJyUGf8KlseoyFrkTHxRRRJn1FRJUHYlm72dCU0e6+OaXaUGJ0iaooloY1QaFPryglUy313zwwtw4Cd3Bo5oXHRQnOUWCgiql4sei7MOUzUYzhoujPUsqh6th1E1ASnzpwRR0iyXkfyacHFTc7RtWZnuFHYRw2iT5zS0JBHGZ6wtHFz1yX/lhoo318OmkkXusjmHMx7ADYlhy7YabnOZr811FEZmXhmPTDsbNLnrzOodMiy6qYdu24whsFSclS3dcOdCNHZ7NvLidci8+E3t7SSnjLPA8OXCn9VS/f+Mdh5B2sVwJjFF1nfBWw6mUxpqNJs1sVGmbp9CKC+YdpGE69gkd5nVnp0jWzN9Y5I1VR7mUprkomaZLr0WNW1K+CnwY+53H57lCM4fV8Syi8YNnuhdM1N0L3xvQSD9zDCQFx5BhPDgi5MOYgxaYNQSz2V3TN2nmWf0JPM0WOdZflATjUJ9ywstLqBQw2uzQi1JZi2S/6Q+GLX0wWV92IQTKzbVKs1TDMt2Kar7OpwdvqQYuk/oCwuV9MIskB3V5+jxKwqxhQdGEEhXcU9nRmPz7nqkySli4f3QBlmGFZwEGlXw+6B/1/2KFv1y5cj0LrnYMl+9ECjKLCcKFuRRBua7SVIaiHDYAklDGBGGjlibD3dMcLijirJ+sGctzE63TP9jTwELPCFpRVLAUQdv0Ixk67PsgdY2PVkGWKCMqhHWJpinDIPdoGLTlhM+rwBLXDd5JalLDxqus8BfUbw1GNLmDk3NKHb/zrQ50M7UJURhW3QXD5r1vvCK5c0RO0by3w3/92pWcZStNeSFANbZdpZYyK1vR3PibTMqgSTDXs4nDUeN7Ai74fXYWabKC+HjWTsvY91zUBNx5PqTki5Fdjy7DYB3KtUT2vWh9YF3sr/jH97F/+nOyR/fvxsMhkp5ur/u4zSF+OBIecMe+c+bApJ6NG9viM052NRVuo30fj3gmLpRNnTL+DO2h/MeiXOt141mz0UuFJodE2Ju+pEkhtmtMCehDGJoCaD4mR0z+5Wb19ailvwLySmmFqmRx+O3KAE/PyD73y/SxZH41sN+GXnizybQLTfhMBQnu//ANB+ebz58tLnZkUQwQBngUTZVgKfxhu5RMmrZmgp+UqXE95W2JqT4cLn4i903l5YDT42Qkkhd7pQY0brmapfj1FSH81YdctRpmnGUF54vq1XyNS2/nupD5rNLjnm5scGc+Rqrzhed/q9plHi53QqVO+s3fu1QWCme5hTY2FiEGq3Lg2FFKoidjs3sZ7xCK4vQZWGFGbu2lekVxd7JUvPxIyIoknRSyzg4gkN9B72hC2+Zh31sqrb1/TLyrFpWaC8yaobN6/04DUIv3YHkZvM3IwIbwglQcypwbk8yzZfkgBB/rHXOZxI7bi4lT/+fndp1ujN9ghYUGBR8XoI8jkKTuWkILeHgXyQJwEB5YIsEwDAORZ70bPhl5khDMTVUmp0+rbs8nd7OppHpixhuV98Sio5BYFDT4Td9V3kTUpGsJI6dTgbIlx3Mmo/jWJC/EyQJVXAW4Mg76H2Fc3l8IrMICpxG+UYXOtsSMnOcBYLXabrmjm2ffula4HzYhP+lCdjt/VwiAxqekheOpcz39kpOk0+FdPdBAoP2cGvia4puQi8hF/DO9y1N7wAowWCJeipb8sIHHmTnGp76a11QvPcFuX/IO+y3XN6XHQKW2Q1StE4Mathf6PJcSOweOkFBQ+0KCO2Al8ItKcwHmaUt3e/iaGkKT6XXIdqNPfvchHTWiazOPzgcug2w6hQp3oc1aGx7RSFpWTtDzziXrTUvcXvFfesTDwX7uJhnydqbCpWrcxm1QxlD90TG6GT8RoKm6dW0l1oYf9MKY7N2Fd67/JUaDQZEWEuycozhOaPNJyDUBz639u4QFd4KkW8s29QtMUiU0v16o148cF+nF3JUqFOHGcebJ5c98/YinxyHbudVndk6DHFmvhedgw0z2qfqHwKo1EBYC+NX1hutVTfY2Q3aGDFEH/y+yf2LkzHM/CkTcYfiAyLPGqgSlE8ZC/335XSdR72foW11PnusVp11/gBeHw9G4UkYg2z16QESNB68Dk8kfZ9/Tg1PzZ5Kt42e68Ow9JUQGzXtaFajr+UgNavcSgn/pHv6oJd7sCsftmxsIkmT+ndmhj7ODpGdWGg+sbwKorg6HmyV8yKor6kz4wpczkZWZO0C2dznerUSQjAvWZdFGNAwo1+LlF3GfBZnVvgDlNDGJLdpbUmWCgze7jPS7MZSwQkwyOPlRnYO8JE01H3Hx6G0VT1mOzttLuxLx+xphPvtLmfKEqCd/yHnwW4OMgcArbNgz8ulsTn8nbufoeRw5s41I8aAlNb6ZjAw9vqNoIFYUFNbuMvMGwg4qBIHG/j4kE1LDhavMrMzOtnKbkcgZ2feYZyi3sXWlzep4VA4HxxHHxC7wBy5Bip6OE55oCkgai/MKvJtLIvKFiv9PyETqTOPhhcHtzm4CzJaO6cCR19+WbM/vx/VDNDgBCVtj4xy+JI5fCqbzxuIMv5KjvPcxi2B9bepkNIDwPEv4uEe9+yCCEyY9l1SI+VZ1QTiWW0KG6sHU+skYljcw53FWExgB06m3egRrAz6IzLq6OAlMwQWKl9VvG+Zrf9Rg/uC7PgNn65Js6PHkhA6aRkSEedgJrvRZO87zrUgjl1sSZwjrktojGnpnuceFo1F6PGEiZRC6VcW6DPlBf65XXKZpqdwvyRUiqITTdKtztFWN01U2VxBkNGoAIqRWblFiyy95YOJDqCOoKNFdTSramD1E1ioVGmGnybKe7oSlq/U78hpuQDqVLoQ2g9d7Cx85FEgQRFOplcHwp17KIhr3+KmijjgE8p8+JWvgCCHJNO/yvlTc+a5uvJCbdLUSSgg0mtFeU1w+cwymfBnOiN8VUDFycwllRmoGYG2CchxY4jAVA3FFLqelU0Z2eXgZ0tLcK5cqz0yKnh0QQeXwRFUgr7z1b/UAyA6tXAi3TgO0VdEQtbSZ8Cx2ne6qwv46BxeamKy9M11BzFO3vN1CXjHjUV8NnsN1xajrgh+KZMbJHD3ANdgYS/vw3FyJLlBM5/29jpykU6HqCIWVN91waeLNG+tfhgUUBeE59uOHJ7jN0yqM3NtFmbORWw4KJyEXpyL7+003CdkP3KNM/PSfoC1l/zskJz21macMFoiwnH2Opjj8nL/gZOcoOkEUL7mdgeoAYHDmzYmP8POybU93PT5fMwOfmv+UOFFBh8BT3dBzfyyPdoWBsdyaQEsYD0LkVc55FfN0IbMLsERLByYmbRzfQhw14aHKeWc0KaBujTy71/XXVEDJ+y3dR/omcHEM/XtIxOKhnJ1BoVbXIE+Dl81p3vWZ2FdODH3IiHHWlUydb7kOv8hdabjm8HMMndYSzDgrD4aVN4oLp1wxPTa9jiUzMaDG5P2Byff3bxhYH4453Cg45O4GLaHU4kgFDKGrgfmuMQGN8iQ+lPOVRL1ax3EHn3C/RLmpD13ZQ1VzkRVFk/jm1nTWZcajhmXtQvvm/5fvJozbj485YXP2qz4jb35Yvz08gJOk+UeAdFO8FPIt2ptiDsTWDhRw6o5Df8TogqxIeePnBs0n40h2xn237om0l4XeTxwLdJb5zg5HMP6nC02ZGnji0sYkeKnjOw575IvOPFWHRvaUOA1ZuUtl+R4D1bBnny5JOoBUgJ/GLlx6/fuSDFrpNYeyEE9tU6x5G9Vx8HDyrq9eeXw1fy9Sr6c+8XeKrmGq3KLcnM/lHWBmt5Pc1TXijbvOClNqNG1by+op04dbcq9TYx8BU5evLAJrhezE6/GysKFi1VTscoNcVVd3VBla4h/gVJ56jbmVPZk8yXckX3x35r9egCfF5G3OoyChOhEjDdlC0YPvhypPPEecFPWG85Orck2LrYR7Ud6z14D4+o7oINl7/SSeUbUF3ENH5wx6s6dTMcLqvt0sF24h9NMYaLz/My9OebaKp+Dc1I48zlul5nPeefBfL6FpZyb3Qds9485rhS4w/LrlbPgGFRqA1U0EojzeZgu5/NytSMwZq5T3QHk7GuosgKyehLXW4iz7+6DgLF8VtgUYJPsrJYemdQyR44tEBI9lImDk70a74Ganeztl+VxJFx+FqY6Nw6alUK+5UUui3NCwBIxQ0nbm33xAzthd1Uy3dkkfTAN+pwd8G7NkX1OdUuau9fj7S58baY8sHTxE1l3eZCsa4cXC7hdcHL3tHIn1rfGuNS96dH5WjcL52lcrV8AcGLCoPDu6osK3qGfk1QVXmfS8aurAGiAqQF5Zit0Oz6hXwBEgQWvOkzgTALDshpb5Kz/sXhRrkBgi35/4GOHi+69vjCTjrldFNchmUwNdYRVz2HHDdbNIsxjq4Cdxl2EDHpQj6HOZLHAewYRCXsg5gMvP1SkHcCm89qRYNipYSjd5YtNUc0sE53NQpih0np8dyEZ50rHfZZSuNSSEaDqAZ2+4E5Jg46VCTKU2L/zuawX7Fq+aOay/FPd8NQVdds2/CZr8IT+7zRVRCchn3dc3vjavna1CoVcNCMRKhcIJtXlcnJDrHMNa7OVf/V/T+4IJpNXAAA='),
    'arc-agi-2/dev/dsl_all.py': ('a88090586143514625a30e38a02e8c9ae3acd5ba249aac45d6c0afde77b8f73c',
        'H4sIAAAAAAAC/9U9247jxpXv/RXlGcQmp6luse0kjtocwEh8CeC1F3YQPyiCQpGURDdF0iTVIp1ksR+xwH7Q/sl+yZ5L8VJkkVJ3z2yQwaBFsi6sOnXq3Ovw1atXn3//+9nnX/1xdif+8MM3C7HLQl+kWXgIi/AxyMW1SDY/BV6BV54b+6HvFoHIk+gxyPKbq6s/BHm4ixfirz98982fv/j+h7+KMBeuiMK8EMlWbI+xV4RJnIutUWRuGK9TN8xyU8zeiu9SLHGjpedGkbuJArFb3Vx94Xp7UWQhvLtIxDaMYThu5h6CAl4o8iOW7t1C7IwwTo+F6TjJsYALsU0y8fk33wh6jaDX3Fz9aR8IKA8ykR3jGH7cNI2w721yhJ53+I4igLFSZ9Dg1atXV+EhTbJCxMdDWgk3F3F6tc2Sg/CSKArkdGSd30M30Lsl/ODnY8DVQnhQJEnUVEqzxD96hSXSIDscC5d6uLr6CkHtQO83se9mmVtdXb0Wsxf+Exs3D738yg+24ivDNRciC4pjFuNr3JxeY7gw3KJKAyeMC5NqZklh7CzxoFSHh7+b82OqtI3CdG/slDr4LMrgYVPjUVPj6Nc1YHHiPO1W2d38iUoAuEmWd0vyoDCgg2McAmihgHs4JHmx9pLDIYlxbJud820SB+biSsA/D+Apl8TY3WTuYxAZ0A6Lwi3UReSMk0JgEwHYTI9i4S1g/SLhLTe7FVWWI/Buum+zzeV8Bf+xK08EUR6IOQ0pCtzOmORQZBfD0Sh9msuZjX3yKgRp5HoBTguWaCM7AvSFWe1uvCStDPMe75c74TjCXcHzzX39JnhOvSTbNcESu/HU9c92p32QwQuwucfw3GyScp1sjYObP6yTbO0FUZTX4MRrRtGmrVoPYaE8uSHMwv43sAUYSh3UU1vXCxMFsSH7g4bzZsy4TFTHqOaWKOemBVc2XNlQkUd3cwhjAwvknVvCXXcBuKUlKhsoGLXFX0a5LEkJh2pQt3WpHsyhBe9uWc0XVDBflPZKYnN4GCIh4JSjoqnZYh9hHkFls+PaG3oPL8JOfAA3O2UCnVFSNxtuvuMRhFGwLhIs31viJEdQwHVxIqzJ924adPAI1gLbYAMgoLTBvCCMjP1tsTcBjN1np9viZJqmMhrEvgV0vzgxCI5pDuQ70FCPIA1c6kteYQ1L4FLBj83I5yenuNu++6bdcrGAmvCH35S6fmemsODOXNkhSGuOUWQYBAgot2D6hI08gf0dPL/DpQGM2Vs1bGDzQWV8dmqf2Suz7hlme4fTvcPdtmvvekB5J7RbslpJD4F5xEFc5B0Mg1klcex8IicOzOr3cA9tAr/TABlvnMQzwDjewcBsa4bM+8xicmveILtrcHY+gaZfW+LHHkLlQRAz3H8JsiRfR+EDrSNufLOzMEuGFfLnCqktsIBdYHwt51AXlW3Rj50iSSHwXcsKNt9KQN2dvEQqs1PrEtlK4iKMj4FS4NHyUTvl+c/wnJi3sTSw1FwBje28zhF/yo7BfUMLl2rz0x52k/hZMwho7pXQ4OebNEmjYFsg8WYqBUJIEPuGwXXkBuv+Q4D4UOgTVAzDtmDfGDP+mVs2/Z3Zpgm0DEvxgY0PqJL8gVtiVYgxzqe8msB/hkPFfzG8LcbhetW1T8O69kttTehy/pkTV599TRwUr8vPfqRrZK0EOugtBtjhs528cRxP/+Yalep6DO6fGyDR4x6MALFaIDJCI+HyhsTqqreZ1odjVIQNg4R9NddtqmYrgXCbZGG84w3T7JeXb4cY6f65lZ1YWu3avt+Npt9Y6gZS9k+zmrRGzhO2jlc60/tmatvwrok3o5gelw6j+TvH8g8cHUF6B2jeEZj6yA0IenCj8JfAAP6hyG8VwNxDiRUB4yFQmhqre1FSqa0trcWhcu4ga6xy08LfUh0CqDu/BDHK6UY1w9ozkNEY/6xS6dDkoeI9apgqFxob7PhAO4PcOfWOMwyU/mCo1/asHrTAR2XzCK5YxunsFB4p9bhAhG/alnWTlUPjVIQTns8OhPqwqFpK4oeZ8xFKNR/pZJOWJkgRT0NGAPGgEyL73JH10TH96PI9DENFIrOj7TekBdg/lyGqqhuyQv4mK85IJgfKZXEbnJicGVMbrlfXUvGW4Uor9UuYAjAtrLMgcYqogkfVsR91XiNkiwCfnJzurBQQTM2qbGf1I80KejKtHzuzysLdvhhMC6sNp1W20ypzC+sMptXfn9sQVj6IvSjJA7/LdACnHLvlOl9GSeLPsDYo8d7DLiMTBZkUNknmB1l+j6gbZCGMA8Q0kOpUpgRvdFo98WvrR6fPn5w+dyLmBGTIYQp+OQMBLvXjzB7yD14FB1ZBpZX4dDHgCy35Q87Rvn0Ev5uBwdu/fj9vHzAlKOqyoistr/nDH7//QR0McxliMgMe82wJSsdbLuUpiKTGDvswPzT+A1vp6BoiLU7GMWqZpJZMZrYUUICUA8L9QBZAQNQCELJi41ttSbwRZMRjI6Eo3Icgb/WPMLbgJSYb6HB+V/xqtBq2hkADuYQ5e0vmyEfgbr5IYgHF3M5C4QR1FMJ9Zi0Payg2trHFVWpVOKtU/EHLiqgNhD0SA4RiGxtoNjP7C5Y1WhH0gatENpHMrdawcdzI6Ng4MmAxnduNTuSW4P7SBZpz1XuIq8j0sfSCtBBf0E+YxIsrbXOa/ilzU5h8S03+DcAOIOVFmG1jJB2JMBg2s7cNqLuAxKbYWW50QUiQY9Dw4/4wtl0MynWaMLzcDx9DHyAl9lWaFPsgB3hu3UNI1th+fZ5Tvt4FyUEZC64gGmvFMnIPG98VIJWy8RLQU/Sf3WmefWxavdUgu6bFxktLWii77VozZ2O9HPTR1pZ1amPpYARNHzDiVQvgsMHgQkXgHqyLLqzJPiZBFfqgr4Assk6jo7p+aLbvaK6vmWt0jdFowcziQDKbLTQtaAwNQlD5+gAo1jFMymG7zGPIcsWXQ/sd/jvAEP72D2UvEittTbxub8rFDmWnzdIlc+WqvyWRSas2YmgAGiDWtodb7rBkFRFVeryCyrCuboEGIKWyKpb04K9MST47XNXza+H0FS8BW4wBCTr39qo1SR+6Fun2pSQ2oKhwcA49oKjm4IF88mCJR4Tq4SYsgkNuSPGELL4POPVHHVYh2R9i4rbBRHWnv64t1SJAOs/odAqLPVk9Z2z1nCXbGXtmxjA2C6jlGrTKvFAwFmjR50Dq2eSSSN8AEDOQeDbCyEEdB8KVHzc5IPwREbg1ZLkgWW3mMM/pBajh78472DsfQ18/3CIBdOdcrW6M64ZFN25cGeawmSdt5zV6z5dYG/XjQi3Z1CVXys7wcpV1EeFTMX4zB3z/e++hO++zHBx9TYbQcuoBnXOKRdfjgI8GXO8MImgWlUzifYq92cXuIbCkt2VpvHKPRfLKooawH6/h0faV9zfvH6/QZdECgCVAe94llM3WACl6s+vNk43pO42zh4R71SivJa+1RX9jXr0IDmi1Zz/MGo37A5Co89NMDxapNzkkMOy56S8TzvIwhoW9GbbuBOn3MV8409oD0J3hM3Zh+Iv0yM0bu7z4VbM/8Q5dQ2oVW6lic5UWaA/CUbq7ve321531g3gr7EHf3frc+cNiQHAPiR+0a/mxRtZDMo+V1MGNb88H52Gh+lUGjYKo06m9QJKchmUQEYWMEu8BqWZY5EEEC7k5FizwoqgeU0mcdgh+91+DfjAIc9yQhS6Wnt1C968xfkj7zP4NsKfTm4euDUYLD1UB3U+MRKMxns5Ur7Wh6s3DwqiubRNHVeJNSTerEU/BGb6piA+4IsQceVk2wOoOIOfSmvoMf/Kbws2mEq0a9n7XY+MCEXTO4FZvithGh9JPJRSHMMuAGNbOxwSGnJxY6l4/kuYqb/Z0c5EnU6IV92X2/H3EeDpY1PVsJuzu7JkWNuVIbekI7fGaCOnLrk+LeUJMSyogT3cLWbONhYB7U9tsz81KTTPQPgbN2F2NRK56g4DalG+Kk2aM8He5QFsmVkr2swqdsPzghJBDq+3qarA95otqfg1tG4rJvu/uM3u14v61hq+cV/sFjOEdMoMujV10/eZv1Fdze/NJ4jD2hpFEfnIElfoWhIg0ChokfCCL1Z0lPrbEJ6Z+RJLqd4bVuqeRPLWO6TcPTxsb7zvsGb1YhhshEwCtD24Qr/J2s2xB+93uWULj/Uj2BQy6oAuLLBN4y6V0p5PMto8O9+Vs+6TbHSz9vdioz+yeOvZQiUazrLl4c3cvHspOsa0U2yudANQjRF04VyqgS0vC5IUS0kOWxGvkxX0lh8xkcRIjHZNMoWHjqN81vFoY2AdFz1j8xBSPbha6cZGbiulmaxtdoRhlRo4qmZs3bo7U0VBYbxuwwW8ACJhtZ3cv6mxHdjAARG1uBl3lDWxblFZQ7dgnkR+0Robtx8rbRnjaUKbAim8G5P8COeICuUHalNEH+sGo+MbCxF4KE3sUJk5SmDiteLB13BV2YtVdYjTM1YhEQRuS9uLWBhT8WGsqegIO5tXhEBRZ1UdBdHVHAYZ0ygrCjRKgDfvbx9sM1Aq3DHLVNlgd1vvLF2pgqHgna8IeF+mZnloVGcJBAD/N7FlHpOsBvJ7b47/e3PYwMVVcHZmb/y87t+m1azcLYafFC8k//ripFvblwS3Dw/EgSVUTzWo+sREFuD6lURNbe3dZs5ss8I9eYCx7A1VHoHR9Z67Ml1KN2r/dt174Xf+0QAc1/EUHGf6yO3VU0PEdf9F1nKPX/IUMVvWwPlfejIMTBoktPZ1tejPngBuytahW61p4blqhr8vA3jCIh26koccStmKaG5jlFkNnMXqKXwgeP4iTMA/6pP/74JA8BiLMk8ilAEIyt96KhyBIRRJHFYwt2wWd4ClV0lCIicqWa36vWvtlzAeZNvvRjf0orJ6dX0YHv9UZ9jnaQHTDOLo0xBujGudhqZGBzswT54W23dH5faqsJSITNWnD4ZviTUhxxG7JQRbSLGqJdQ2/fBq8+RQgMaAr3L0XWN4NgXk3hplBWWSuV6w58LWPoN/xwQ4Oj56h4RIFYq4rvD3skBh1bNhcD8IgXM2L2/wAo8CLoPBa0ZgABPAnuvWJJT7tqf5eFtK2Nl7Jfl5Z4lXdFV5nrnRXvOohaGuwxeXFPxb15uAfUxecegZD8I+pY55DZBnYeGVNmg0scTOZRYsdjFLUDyidQeVIAlQsCD/IHTZicGy6bQCj9hvGz+hX52sjOMVF9zDHkmXnlXqoY2CzwnFAS/0ZC31IHoVLFa2DhRcIpltII0LmrcZCBDlcI5pekOYkRQt+aPMEINV7rR889w7cNbUPblOt0QI/ZBHseqvPgKExkCIxyFyPG+8sSxgIkmc3wCdtXXwN1s2TDDgUeXtHieGTqGHq5vkYLdNpT8cNQn6dp1E4IFM/4EMKTwFcwLNqBREbhBvFXMjG+T0OZBNyCAlgAEBU1ayoc4AF9NLTugn54fFVJxou72qW7P2xTbOL514SaSrN1Uqv+Ywdhid2ho/932J75Y0gF8Beqfq6QOckBWIhbNaKNg7O4wYR0lTHxN2UfZ2hc/hCdrOoVQVdT3KZeFyW7PjJQt+W1iId8cZhpBF5cXltsGMcSp+FcyATE+hcEyb9WkhGx86ZXDGZ8SNcd35bf/1pHGgjQzIGAmWGRzGXLcxXgFedYk8ptlcq7dppjjAgCMIWABT7uDEH4XUSCYbt6z5+UvvwRvro2M6X2WYZrq7tBf5e2yuY/mb5Ez7A3+v+4DujqEPcBjZxnmRdjPGZOhvgrl0C6CEnGzatAq5wdxlamgIyMk59Q1NFOMBkqS15Dmjq+HCLtkigWt0NhjFozd5/DF3gesEBCN8JhHKRADEEZSHwc0GRZIIWLldwDMGJAwAB2BF3hGu4GaTVEzeIDGSRD5ohUaOhop4Qvi9b9ZIUR+DfdD3mEZK8qrREtWiJSglCOfoh4Kn5xKbc1hQfCqOiK+pmbj55BACA6rltP6C2HQWdZyKUoIOBsJekTpIiqtTHsuoLQBpnuH8b9MugqtfQk10PydQtokG3hlJQP9z6LAJqZJbNBSILs+9NNCY5Na+FKvZiNe6c5CHVUTZ4pmEiSkw/iiQ14AJDz66e6I19nsKcuSdQ7eIgH5pL6bidjF4FbSR3D8GMmH2jQVMkFLV+unCksTzshhqxJ4/Jas/jVEC6ylw1O/NxX406WLEueDdcvYrPzDayF9S8p8OxZedhmWs1lWpOW9KWyuNcHpq9tlUdUlEuSmpTNm0WFdTHhmNtxhSHnnDC87i2zVaTHelSY8bk+baN7ZfbFHTmK0xvsOYYfwXhLpep+yeOa3VJSlK1zwYjhDgUyu52uYRpLmhefLx8Zqv3IIfN1duZ/e7B4ElXxFpGqvT33pd4MmI+4012xChAjGENEz/0MJAtKEKKZQ3jbZBldTBrsQ+QLqIsO7Ebn2YKlyJzrQXLUSDWUMiHH2acJUKNOy/DXB5dEH2xCM8N7ikQCiuRa5sW6TTg3F1BFbiWhs0kD/LIqlZG68h5sZiJMT7lknQWrjRjIqk81LOEDTe7TscbQpm2KfkWXcZTFAg2fDnGxGS0WyN2G+7ywMcU4VeqCuMciWBEPu17sckC90HPNsbpSxeME+/JyR4Ril+JdIo7NoCajlti4Xu3zMcMEg0s3RwniFs1ZE/L1blYqHC1xFa4qeE1fD3aaJz2qofQaMWfMlrCrMsGTFWbMcP7JsasLvG7IFW13fLgZg89jwO69aWyJxV/wbXQVOkCcciBZXuYXEfUYgOVZ0CisjhoT3NpXov2+nWR8GtVVqFjK0k0GNqX8Ixz/LCb94BXiH6sniBN3bvRY090ObgPgZGkwzDVHu5PhKGdI4GXk8FajMUgqZFQ/5aELeLb27tRasSlo6QM2ChWAHV0tVwskOWN9NPWwnwVs34ES3OeoXsWaUyGlUcstPXGDwRdclrhzCmG7dVl6mGtSbnWpqNIuR84cwseDRUwbXWqb35obPBHUb+oD1X72pLVlFHwhVHZaE5ey4QkY94GjrhBZisjZrl+4N+iCyKFDQsMv3FDYI9Te5bjrB+DLHIHwRd/AD1DbPCYJ248qChcPvRJcoR8wy35BEVYPFObuMTr9mlvN9LJbwvEXZkFh0PFO2fQ+4GCTV6crt7RFnKZQ++zRP+5+U5pc35yUySSUhTuG2uDAo8GFAnJy4IrSWstrhK87Bkm7UHCH0W6Jv1r/h4EZlgPtCGtgXK7w9CepHGfUfEtaBfAdfwmJ1sbXUYs2AhudjcYXQZQAFU78HfBdGxZG+3VUTPn9WaWWY+am7eSYNpN1JMS4jQVAyG7J/RRwayz+OiafqBv+tLwCPSUy2Rbzw06IGe7ozvV86HmVI8aZ4BtJ4MJehYA2nzzF5JQEH2C2NdbR76gMmBTJOTsyAgCs9gdKJWKC+Qzz0kjQwnk4m12r0uT8M8ylnyGkdnarjonEqGiPIXItC6n4OnF0OxAlo+6Vam2QucHNnz3dCPNEp27/d/5cePsS9B5ReKO0biDxB6WNtll7gETI0oGaU7xP1zDNTDMER//Fx1Otwsw4Q6++RQ3wnMW7JBWGbKOdFEpZ/7GIoL2STTEUbIiUAlgTh76gcJsyXbn1udhb9lLR8m1uokW/p+iilAG6YYPDQjAaKCREmU0caRs4jCpJovdQKBvs7hopYvNbiSop0k2sizrhC+lmvCFy+xB2YiZ0motk0rmF0oGw9ndSiUbzDCNwpGNF2qSvxFdVYlqhpbdJE99GNVJvjoRNdjL0qOMOVAyY3ukmgqgY2rCrCAsAnZSgZDp+bjRZoMYxCe1I7zvZGvSD7fqmE6hXevZHTeGlHVCENE2sFejvr/agoNAqNAcK9NctMk6+OFCuRum7KisciwHVFnqJmFPTaKqhpOY/9MmMZoaayyz3Oi82ixywxwlz8v/pmQw6QKrk8ykC3R6TIDq5DTp5TmZtiY9NW0WmSKIwju8Yz8UlPBEiyt7mOy+lMS3yXJKzc2JAOX59Z5cCdf7csjWn3WqXsPFEt9fw+0aOhqyMYAiilI148JwOPYd05l0yhYkNYCQtNkjlMMVsG2EycWRcfX5VuMjafSiEF+OC/roglg4bO/gn3caC8d5pzDshyWyKVfia4GplFK0vzWp0nwG1bX+5OVYtmeYn3uMCj/0ho5GekdO1LWpZCBP1pOoM/FJaq9Lo83xxko4Rn+t+ikQPc0+ABRCXvpY81lKDsG93jy60REEJLMG6COLn6sxgENfJprl7Wl4d2LdoIXOfdw5vNxg1WLSszsd+nZGlCH65nF6jlF55oLoV+gDNVqv70E744Z+pqWKU2OgvSr/+ehmwai96tvyW9zfNvyQxNwkzPAEETPxLW0x6ImkhuQQFHvQzp4lzfJR2865+kYL2eNqQhHaUU6ER3S575loYxkBKc/sKqTmhHndidTEsOBIZ2LOCv6RJoZlLIHCgBbFcyeWOS/wBQ7+GaHsMA8aA+IlDWGCX8aET0yFRlOLjoSpTjtQOl3LlOhiJv42/8eoM6yG/VlzcJMiGQkzUFYyCIyexO83sqlN+0Jbph3FzuyJvp6F/+ziIDOD1mpKpoVNUJxQ4EU+WJwS6Uhp88Zo3C3PwnrMoDSqw7m1DkcO914GF2iostPIUzy4A9TtKmsW1HYiT4OsadFPy64zW3RINzQg0n2GVZ4hoto07ND1ZAzIhAwlNbO5BT0tSDejS44Tic4Fiox2TCogmq+5U75sg0nGOnY3OUwLT8dTdn24K208Gj+SRlY4nRYXeYqvp7QJhgjo8eEbwyYA2m8raTPF3Hs477awtN+WbeHorMbIzBQOvEsuVoeXS5PSZTHmFYV9L4Q8vDB768HS1cEWcHcngsJ7Hvc6FhfYYnr7eI0HcUICiNc7NLKmUQ0L1Y2MHTj016qbOPL3OTLxpy8WiM/s8vbg0VNPczA4nncS5AlCMclycl06Ge3rU02SLTWr06/Cj6mSN75xJ+XA97xp0N249sNsuFWwBNlaULBd8pAUeFIlYQNRbVqXzn03JvPte86N9q9kvZ/o67XYJ1n4CxS5kfT4njXvkymjfudNfjwYuPt+1Bg7YYzkBRiUlEAUKZZSPeVS5uboNm+acDi4fo/ImDSnfQGaYmad25Gw3lJpM1KJ7VOlfrK6g9zePdS+duSwxgBbTgP26+G7eLS6YVZ6wFYTgK2eBNhKBWx1DmaV0mYSsJV+siOArTSAfSdBTnwKed1PX1eL4BQQQpvFK1xKySht0zKIofEjoSDuBVnhhvELhPAppe/dSdZneOMFXo+xmIouUYOmQFS9EVLU9W30Pkc0GojRF6ybaIxIBu2CnGorjyeaSxeI0lw+tGdjzd8lCzy4PwExLqr1Y1IEazR/4Nff9Afv6IwdVPEzzIZzW5+2IxKCuadF3dktcJkknvIa1mF9IX4qbiyqr0nBDniNDFeKqYKiuGfocJGYf5GJdWgyHeTzf7KN9LKzwi84EfwUxCRYStfatb2o0RAuCZd0dnNuEm75glNDkrimzOF56UmTAJHml4Hvuw4k/5OsQALVopeciQ0L7i4LAvIUX8lY1wQn+OXSyK5BwvyVSFcW3c06dxldoUnsS1A/8PqeZbYTuho4IIK6+z1GIeXCD90dfhxxlmM2MU7CEAUUtlO4+UPO4Tpi/uu73/xu/hu7dYaHfrmNyX961YuIychEky6EkeE5QRqbNVVpdrZSNl3stcWtQkUDxK3AI1WJuBrf/ukYCaemDv2FFzm6MPadNJDILOwaa++ZpIm9fNU6mVNih67zvtD5ZDW8q4PIj2jAe0d4CjqlCRoGn4NLR22FlIzuwD655QNltZLZqJ4eTNpAajmZM1OC4oAiHCAFpf9iIQqpFNoWK1JoMhAoAv/JgDqTkWPkWNB+yqF9WS7RC4HeAF7GeBzORPlfdiiuEQAR+O+VF1P2ujjYuUWgDd5pSaMIypRyzWPcUCzy//nvnM8stnSUqlF2OyRnGBdG3ki2zjA/lXnwZOwtvDEuKGUqFWL7j3KmufdC6TI4pEXF71Mo6fw3v7sL7E89oKQVpmlE6vk8lRgIRniiPOBMMu7H3SHhnmPEkd6dmstkDwQRithHApenQTRls+45prHCrqHR7S29HNrg1al+icF1yLBr8OhM3l+cA5gY5+EIsgmfLGbI+uEhH32r6vGFVUcvTJooHpg2OX3t2oVR7B0ejJP3t8w0IZ4gwoNMf+INQeWEv5jvjz9iO8j6d+Fuv3Cne2eyAbc2iqvx0zATQr4MKZLRs6biG0X4S2vWbuI8FJEDAs7CqPDrpqaEVEmQWhhl85DyEdIr37ynmIV8H24LzbHZHHTEgr+OEbl1+DEa4uhLle33jljG2lTC8PGzyr0cPn7Wrtisn08Vy73xcjkl6KJJHuuPLZ12yRrE9zOHB+f43rNEjwt51oWyRhO7U2HuBxxZSRdDHU1GGxtxhRYOPmkXo7Whm8IDn5Wd8rJTrjvaItUN7Hvq4N0ZHr7kWUhBgib+fvlaHoU+naYqkqO3H+iWWCrwywEc6ZLXgTZhTG5FecRTHAFPIuH6P7keMjPjkxnpY8wC8Ts3WaskBLVN5EZ8gwGmwsADXTK41Go7NeXRVfyQeRsljQfOQarH7zJRqgSj/p1b7CDiY1WtoH2YMIjwXrLkfvEHn0JqabznHLynY/yY/USL0bgHD957kKEJ/nknay3iM13qXocwWeuy2sBe2p2LIbx8D9bMm/ZdcygbI0V5V3a2Ylv2lzHmwttT7absdmPX3ZyV73faCq/FKTlGvuAjTGmN1VLtBbkrL5JUbAKAXlBXGg1UoPWod/qLhtXQCDF2WnTXoSgH/TK02Eg1R+YvN7dXCW8feA96q7DrK7KKgjx33bTSeN/ILvqwYdnh0l7gXrcXbHKT20R/2NH/CSVFRvcP8Stf/nK+mN3Vrf9O/d0tmvuzH0hQx4B91b3wk7vFalTHhNE8e221Bc875SBNOmPpL79CR1mIn51rsgjIJOlekmXHtOAju9T2Rojvg80xjDjmsrYo8icK0GJvpHv8tkKy3eZ0iCH2oqMPHfzvf/6X0oJru4WsiadIUTSnUzHye3XHgCsdU2IiggQqwBioEbPGg9JDeyKjwIglP8xbs1TTGbrfQfDKjsRWFEEq3beU7k7DHNLTeLnCINK9g19PS09OenofZhhY+z0aDlD6v0P6dsI71AHuns4QCLDdfcpDn9YlNPpEeqH5IJ2yHzy6lJ4NKNligeMo4efUJDgc1yY6aRGxi5sioaMfpqlkPbRN1mVH+0FYaBJ7NrQxrCOMpAaFB5IQqBofplr1ZC45F9JiWHWDlnJn3OLlXwpk/1Io0x6AV9J0DZgVMOWKzJoAcqOk+5LuT+Owyj3kbiBHyN5g1QDagRsbkzYfmm3nu5HczVt6DkszbQiSoDKokcXz0MpI/bfI3lEsuPn1c+J/GGDUjSYYmOmMI1pQjKg3VGFMEESJn3piMNomD1eekptr6wMCdprMZRN9/XepKcgjj0C/hvmg8EjjLX5ykT0/FI+xaExVrWIApJwf4r5tUsWwyYs6+7SV/3Okrni+Eqg/vLS2dgXydOUjfnAVn+N650WA0leV8HEA/hIhDgI0FuVkueFJc9jvfnO38Xy/aw4ztVpG/c35efvp+c5nYJuv0tefie2qHuRIIxsSigo62xHVcOjvOXvR2KcOx/ywujjcsfNnYx8c57KRT44reFZJ/xlqKbP69BnaW+5FVsqysikruUy3jaEnUEygDSZ8q+rsFnN5+1b8dsQ+NWZcao4JGe2g5EBNtCMCVzfaIclhcslwliTewjC0peeUyYZywIhcPmMkdSSvHNeRYAHXlHNxt+SGlH5jLj5zoCPFZDGuHcn34rkkaqeaMti0Nh8R+3G7szgVgxiWHXhDb135ee7ujs7YSnDMJhz+a44rnzQTNhPmtRvPu/egaPa4Otdyz12LqSBTcig18AcZ6qFZA7yZPO+FAOzBvQFsD65/Oate8Hetz30KoqPs1XIKwehqUmN4fkaflDLhr/2kGMYa0EdHZ/x5HqyA63CgEDuQ6/eHACiOSLME+FOeAw2/bzYmHrriY8X1eWCpO7PpKArcR6TRTexeQ6g39m8/vvt4oyfUiAXxhIGnPf7gObH3HBL71EBUJerg6lKJ4zXZgNt3sL9iA4yNv/LaxPI3pjilOWfYGvTK8fGaPLdDnpEvRsNpOVX+B6Mphs5kYcexSbFM354HD2MZSbfOUUJQ6+mvBwDUp8OKYxrJY2PDSI7u8YDPdBrV2EvwBRjK1kMaNK5jmUy5jinm8OZuoImg5b2paHcr2qtnWezHBipJnnyViuF73Dq+9vA1B+LhaVON6MBd1kxshIfJ/WA8gXCOZUseS1F3noLupoKWVRgMcgbWhJUYmKz24jCcH7775s9ffP8D7k7+vLvV/3a51f80tCU/K2wNP6ur5H5pPkVryS8PWt1vr1mdj2BZ7adtrP7nY3pdSuORNfhYhdVPC28piWN73XRTfFqaXJeWzNdm9TJG9bpRcxtZwxxAlpKVxurla+kPqkmSYamHjS3tAURLPZbV60x31MNSotktNabU6gfc9TqsQ7UsNQbBqn2M1sCH0+ugbwC0VFXO0nD91dX/AQAXiUgSlwAA'),
    'submit_ag2_perfpatch/out/arc_loader.py': ('2c62526daba3fdd9da317e3dcb689acca8e24018e56e4c7f2c33ed15381ed8c6',
        'H4sIAAAAAAAC/+0aTW/juPWeX0HkIinxqE7SyxqjQRdot1igXaDA3lRBUCza0VhfK0pJXI//e997JEVSkpO0HRQ9dA4ZWXx8fN9fVFG1Tdezr6Kprwr5XA9Ve2SZYHV7teuaivVdVotd01W8E0wB/Tj0za/NgdfFP3h3dXWV8x3bNvUz7/p03xV52jep6Lui3vv4M2CfvjD4vbli8K/nrz2L2PU1/QLMrGteWFEzBJUgemHLyxJXAMAsjDhuI0TqF3XvI2AQXE2Xr/9ey1M63g9dTe9DJKz1A0l1IdLnrASKRVMOfdHU/n7gQgQbe1shilr0Wb3lcnUFwgnrPOu67BiwrM4ZvYZXRcWiiN3Tu6ws/TX7zF7Z54g9rImhV+KTgMVT1nJFhXgadruS536e9VlaFqJ3CYDjQAt5U4Ut76qhz4hSAxz2Df6vmZJAPK2a3M9WLOdiCyz3TbeC41FJ0U9ZKbg6w0IJaolRnEVA1BZIrdnNip0UeBAWIi/2BRyYEIpMCMDKBNgGMGEhDKKICAPq99y/WysdZXAQ8JQJEqGfybeAPiMZRtGDUTe8rZteEb6ZUItIuj2e65w6bsaDsjgMw5W9URLNQQTmGMWCJuDePv/fP9ti0oaJs8RWb+aqravzFIwn9X8beHdc1NKyQYB8jSVY+D2F2WO3zPPCr01R+1XW+qBMRy6gnaurbQmiYH974fVP4PRZ33PwW8KGNKZpURd9mvqCl7sV63UQ2LgxITCCRcBwhAPaexM4NNZd1afErEIrGXeiBv6j0FLU7YDx40K8oa3xOomvCfA6MQpR0rj+/K2oADzr+m9fBtA6BAkQjIX8VsHwOv/2xYYGyRQYCHqMKw71HW9LTT09z6lXx1+gmzYB3YF7+uQYCMagOyV7fF4xUFefQhjbPoHNcHAz5dyz88GQZ7DTqCpQsoQ4/nSXuIv4dlzd2MuuJ1mYfmlq7gblMe7rEE8RkXC6GD6i69cFJY97m6F/b7MEme7WyWO3YCgnQ9b5XRM5WXScbZVaGiFBFYJCHErrQppDHzJOEuMuspRxQdofLRi+5raPCI1FVdlrWvOXlDxSkFlZjourAvw0l8hl1AG3bYQfP6xXkNISyC39seUR5Awr+RI2Fk08P+T1tsm5PyV5ckwSzIgGS/UlUmT5ztCvNSsXUbcy1FqxCXJ1WVQQsaCEENHDOnD9wcL8md27wlenL9nwhLOcE2cSE7mGJY3u6KIti5qjdOxaJBRtCdkUyxXXFh+PRDjmZSySrGIppkT9GhgnQsTI1KuVnWlVLkgAkWi+AU/gerhaUGcG7IslOpcHlzT1FG8MuIuYtKIzJipI7Vg0H51zZ4WZLLfmhOgkissmJL1uedu7wC345tWSdnXS+7Hb/hFqKsF7lfH+IDAxbivePzW5CcRN95J1uS6vDvy4YhAeUsyk0a/dwF0jy9C/ybdNvrcjYNOicgCLsgIv9IL4bpNspiJhABpFXtf0P6y9jSsEVWzQmq6nTHiGzbSVavm2EdzerraKl6zNXrlAlsC77xZxhBTlxEvRP/ljVRFsCMek6mzaSbWJLGgxUcqw5HDpiG3THhG/IRPfXGDQ2QkxUG2UReC7G/irhv/ghm6o3zpBoL6zAtj8pel/rtqSV7yG8vhPXdd0/u76Z5SNwHKuQdS8k7Wdd2ras8eG+lA3L3V4PYuG2VumKQXuWCa8+t6WGW8wxv0H9gmkRQ//Y0ZK4lHV/v9t9LvY6ELDgAVMwYUskuEhOp3JTkWEdrjCsN90xd5pUJV5IZRTKsk3kBoPZK0HZayCgG3IxG1GFA2wUz9Z2BFeav502GiA+JC4R5xdjIoZwKifLmJUADOMiuhaA0xOUHKBE9STu6wkoRpwRTW99YNgmRh8M2nStOPlaS7ToC7hTcX1hI18SoilQvER//I23ZXZXsw0R28BUV5se1+r1+ZJ0acRKFGdpzZFW9KUMnWaah4jW6PGqmytKPuShN7c0CGBCqKEbZreu6ZKd0XJ/W0pRoulN5apWhyiq6F31L4NG+AQbecGSWN5SF2W+zPPgSP9pS0RzujCsslyoU8JVm4AVqLFOOauGPbH14FRKeJMlaisBnZkw3DQdlhy7q5vbm7YX2AX06WZIJlBWLA3nr0wtEODkZKDXkqJGQ9akIwEb7MOTBMgLFGoteCSO1oep/Y7jjf6zqK1GRlR+kuroewLR1SWcAALRDowcjo39g8Qy4KFk8xgTc7EsOK2bRjoiz1s4zzohJJ3nGCu5njnnQ7n9FRAfKbDV/Iwi7xktWhgJ7Nzw04eteTwNKeNFqD1k1RuWHyB/LhIkvMlIs6rSa+lsoFNRGwrExEDwkv4xvBp7zkvGvw4bJ1o8DvFGQc9qTywDs9a8IDcv1EBVszP1yvxOrmk6VFlYN3PJJGcJrVqp5bRM73URIdFzytIB5ck/yFcCvoCLrJvxiJIyKqCXMBFsQpxKWj9L1lUFtRpqSjqfclVbMIXu6HeTsbaKgWpircdeoiGdWm3iyb2OrlljBTnqwkb6MaJUw8f1o4ru7EdyYEtviwIyQGL3UgulSOyMpQJTi+Ajuus4mlKadAettvJ2h3CW/cRthBE3wVT8ElsWUNQCXc01/WLqHBLVKIzYmVWPeYZw4uDjT3A1hRDZfsWqQYMEASTgfQo4js4CMW5G/Umd++uT4f1OTx5P3uyFh9VKdc973zCk8/XM8sLlW8dJl3ByPxdgnquILCc+g3zRyIBHXHUQ/NBB3oMmJKXDpPjMxnS+xX2TjWOWpQjmDEMlzGqWjE+Wosl/3HHtHuyjEtZ5dKoQ8VCYiV2OcBTsykOPNLJInp6NY9ws+A2r59m5Qr85/jqG06qCvw6ukNLzbYH9QL1L4tG48P6t4ql+ideXMj6xtbMrOJUVzmoQB8x0m2cRm1Y2A6dEsZEPjGshGMYJ9NQv6QRwHKiLVe+seIE3ZEBd6a1lj+juce7nVoDvaATbawioZ6MvSTpPv43nkeEUEoLraB5IVxGVsCMFiNnZEfR6VxUu9r7ggreTq8a0Q2s0n5UsdzsmNae96bfWLHdeDPlXlRZYtLXFSNo6N6fGJ/kR1PVBE66OM72m0uq6X5ZsTmJZb7d3BIZ/4TtukObOq8OedPxMxF7S7TcyoNgv3yQQfRf4nnpComGIbPKBDs4aTBHdf0UqUsoooUiyFHdf1H00CYVWRSviIsI/9h1kaCbbKXkD+hX0SRrRDQP1zBU6W3JFONX4lhUiryC733AsCBugcuu6E6k2aWmU14aVIxIxhgwn5QhujHdjOM3MVQ+ZNy2g0iK0y78RMB/DiQzzw/zfAJ6DJ+zcoB+RAHdIxCA0g96vh8hFiZSig5S1kfpmOcYtOGF0ZIKxCSiNy4Lx/owkuqc3UrMpD7PjBh6LswwTCC+OOSglUoyOwJcSE2ua0x50AHtwlnB0g0TsEdtoHHb2aVZLhZsPDLWHqMeE6WgXNhVt+Vk2wHvylLdclqobPuG5ZUcggAjM/uG6qVN8/F6bzoQCt0yAW8X9ZQucd4uluO4sFiSLznzZtbHHOlajegbOxUlNlfwL08F5CjF7Ge9xQoJlwQt5RThn4WbKJ0zNELbTZcsFiuUyXT9010yn/jO9qqSF/ML1Mv8eA7562n2ZYcZMMiPIsYkB/8m9bOeYBAlWJo7VM1AlWOPOxbJvlqmGBBKOs1uvE1gt1DPwr6TgzS+p7ViNxrkOLy0gB42ydlL5gcu6UENhXz/OV5G/Yw3HDIjy8sIEJlaCZxumIS6XM4r/arcfOl2f0qmHU7VKPnobNDeNHY6U7u2HEtjedcWF+m0PXGZnuVpCXUKmsxxdhpZdJn2wjpCcjGd0KT8VcUpur8y7UIKnjsdwv73Ig1WlfOp3XIVSWPTi99r1WRmdNfktND4ZtYeKr5dNSHuNt7oxWTJXDBKeDpK4CdgvveJmmu/DTH1BF9+CHSlGUw/Dhvb9yB40xTV78vmqABG/xu/BXjWXwE0j1/5tg/iNhkP/YAXzuuhiw32vMB27dwicVbefHBK+J6hq15aS88ugIc93r0pg5dd8tOuJLCxHeZQcP7+3rb60a5wzcc/RgX5tL3FF/mlOsetaxxQurVFoh7egbVuV1ezzxkRQW16+pE7u86aIrUiwbwdcYt5MTxWhcDLyvEWQwxlP7ut0d+qWjdQkRoqeH8dRM8eofccaoaXnrBW1Fmpa6nQdGLmNGXQ8WnnYYlQtX16Km7vaIAdr9XE2url79WwKP3oHcCb1xUUvYlPNzDIr6wKELslFwW5sqif9+fj0pufwHwA8+ziyMOLoz/zmq6R670tQ2TxRF8lSUQgI/kRmwjDcDIX0Z6vQLW/u0H6EfSVFmCF9FDjuOSgK5p0UptA3YIGhO0tesxIVayQJPSplUI0+WaKlLtie5r81UOFvHGsLejTMoM4SBbKOGs9nloPRqG99V23ljx9FAVnzO19UfTfydq3TceBoPWiIuxguawNJaVuIqUFkdAQUwqCvnIbpXLnAcX6x72XLJdSWFSrxJLy34as9DtbMPISKVZokguVteH4NmJ37HeU7Z+Di7CPHc8OMz9CBFf/BFlB2b5hMQAA'),
    'arc-agi-2/arc-agi_training_challenges.json': ('...'),
    'arc-agi-2/arc-agi_training_solutions.json': ('...'),
    'arc-agi-2/arc-agi_evaluation_challenges.json': ('...'),
    'arc-agi-2/arc-agi_evaluation_solutions.json': ('...'),
}
ROOT = WORK / "repo"
for rel, (sha, blob) in FILES.items():
    data = gzip.decompress(base64.b64decode(blob))
    assert hashlib.sha256(data).hexdigest() == sha, f"corrupt embedded file: {rel}"
    (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
    (ROOT / rel).write_bytes(data)
    print(f"{sha[:12]}  {len(data):>9,}  {rel}")
DEV = ROOT / "arc-agi-2" / "dev"
TRAIN_SPEC = f"tasks:{ROOT}/arc-agi-2/arc-agi_training_challenges.json:{ROOT}/arc-agi-2/arc-agi_training_solutions.json"


In [ ]:
# 6. base checkpoint (public) + drop-in self-check + selftests of the embedded code
import kagglehub
BASE = kagglehub.model_download(BASE_SOURCE)
print("base:", BASE)
sh([sys.executable, str(DEV / "check_swap_model.py"), BASE, BASE])
if RUN_SELFTESTS:
    sh([sys.executable, str(DEV / "check_sft_data.py"), "--selftest"])
    sh([sys.executable, str(DEV / "gen_synth_ag2.py"), "--selftest"])
    sh([sys.executable, str(DEV / "sft_ag2.py"), "--selftest"])


In [ ]:
# 7. data: public train (held-out val comes from here) + synthetic sources with fixed sampling shares
import urllib.request
OUT = WORK / RUN_NAME                              # persistent Studio disk; survives restarts
OUT.mkdir(parents=True, exist_ok=True)
DATA = [TRAIN_SPEC]                              # first source: the only one validation tasks are taken from
CHECK = [TRAIN_SPEC]
if N_SYNTH:
    synth = OUT / f"synth_{N_SYNTH}.jsonl"       # kept with the run so a resumed session trains on the same tasks
    if not synth.exists():
        sh([sys.executable, str(DEV / "gen_synth_ag2.py"), "--n", str(N_SYNTH), "--out", str(synth), "--seed", "0", "--exclude-train"])
    sh([sys.executable, str(DEV / "gen_synth_ag2.py"), "--verify", str(synth)])
    DATA.append(f"jsonl:{synth}@{SYNTH_SHARE}")
    CHECK.append(f"jsonl:{synth}")
if USE_NABIDNUR_410:
    nab = OUT / "nabidnur_410.jsonl"
    if not nab.exists():
        try:
            urllib.request.urlretrieve("https://huggingface.co/datasets/Nabidnur/arc-agi-2-grids/resolve/main/"
                                       "synthetic/curriculum_v0_verified.jsonl", nab)
        except Exception as exc:
            print("Nabidnur 410 not downloaded, continuing without it:", repr(exc)[:200])
    if nab.exists():
        DATA.append(f"jsonl:{nab}@{NABIDNUR_SHARE}")
        CHECK.append(f"jsonl:{nab}")
EXCLUDE = OUT / "exclude.json"
sh([sys.executable, str(DEV / "check_sft_data.py")] + CHECK + ["--write-exclude", str(EXCLUDE)])
print("training sources:", DATA)


In [ ]:
# 8. train (resumable: when a checkpoint exists the same command continues it)
cmd = [sys.executable, str(DEV / "sft_ag2.py"), "--model", BASE, "--out", str(OUT), "--preset", PRESET_USED,
       "--max-steps", str(MAX_STEPS), "--max-hours", str(MAX_HOURS), "--exclude", str(EXCLUDE),
       "--val-from", "first", "--fp16-guard", FP16_GUARD, "--log-every", "25", "--save-every", "200"]
for spec in DATA:
    cmd += ["--data", spec]
if (OUT / "ckpt.pt").exists():
    cmd.append("--resume")
if (OUT / "manifest.json").exists():
    print("manifest.json already exists in", OUT, "-- training is finished; delete it or change RUN_NAME to train again")
else:
    sh(cmd)


In [ ]:
# 9. gate: stop here unless the run finished, is a drop-in and improved held-out loss
mf = OUT / "manifest.json"
if not mf.exists():
    raise SystemExit("training stopped before the last step (time budget or disconnect): Run all again to resume")
M = json.loads(mf.read_text())
print(json.dumps({k: M[k] for k in ("steps", "updates", "train_loss_first", "train_loss_last", "val_loss_before",
                                    "val_loss_after", "val_improved", "non_finite_steps", "skipped_updates",
                                    "fp16_guard", "drop_in_ok") if k in M}, indent=1))
assert M["drop_in_ok"], "merged model is NOT a drop-in: do not upload"
assert M["val_improved"], "held-out loss got worse: do not upload (lower the learning rate or the step count)"
MERGED = OUT / "merged"


In [ ]:
# 10. upload as a PRIVATE Kaggle Model (first run creates it, later runs add a version)
if not UPLOAD:
    raise SystemExit("UPLOAD is False: stopping after the gate")
meta = WORK / "model_meta"
meta.mkdir(exist_ok=True)
(meta / "model-metadata.json").write_text(json.dumps({
    "ownerSlug": OWNER, "title": MODEL_SLUG, "slug": MODEL_SLUG, "subtitle": "", "isPrivate": True,
    "description": "sft139 + LoRA SFT (merged, bf16). Private.", "publishTime": "", "provenanceSources": ""}, indent=1))
notes = f"{RUN_NAME}: {M['updates']} updates, val {M['val_loss_before']:.4f} -> {M['val_loss_after']:.4f}"
(MERGED / "model-instance-metadata.json").write_text(json.dumps({
    "ownerSlug": OWNER, "modelSlug": MODEL_SLUG, "instanceSlug": "bf16", "framework": "transformers",
    "overview": notes, "usage": "drop-in for sorokin/qwen3_4b_grids15_sft139 (16-token vocabulary)",
    "licenseName": "Apache 2.0", "fineTunable": True, "trainingData": [], "modelInstanceType": "Unspecified",
    "baseModelInstanceId": 0, "externalBaseModelUrl": ""}, indent=1))
sh(["kaggle", "models", "create", "-p", str(meta)], check=False)             # "already exists" is fine
first = sh(["kaggle", "models", "instances", "create", "-p", str(MERGED)], check=False)
if first.returncode:
    sh(["kaggle", "models", "instances", "versions", "create", f"{OWNER}/{MODEL_SLUG}/transformers/bf16",
        "-p", str(MERGED), "-n", notes])


In [ ]:
# 11. verify the uploaded copy and print what to hand over
import re, shutil, time
ver = WORK / "verify"
shutil.rmtree(ver, ignore_errors=True)
listing = subprocess.run(["kaggle", "models", "instances", "versions", "list", f"{OWNER}/{MODEL_SLUG}/transformers/bf16"],
                         capture_output=True, text=True).stdout
nums = [int(x) for x in re.findall(r"^\s*(\d+)\s", listing, flags=re.M)]
VERSION = max(nums) if nums else 1
for attempt in range(10):                        # a new version takes a few minutes to become downloadable
    r = sh(["kaggle", "models", "instances", "versions", "download",
            f"{OWNER}/{MODEL_SLUG}/transformers/bf16/{VERSION}", "-p", str(ver), "--untar"], check=False)
    if r.returncode == 0 and any(ver.glob("*.safetensors")):
        break
    time.sleep(60)
else:
    raise SystemExit("uploaded version did not become downloadable in 10 minutes; check the model page")
sh([sys.executable, str(DEV / "check_swap_model.py"), str(ver), BASE])
print("\nHAND OVER:  --model-source", f"{OWNER}/{MODEL_SLUG}/Transformers/bf16/{VERSION}")
print("manifest:", mf)
